# Diseño óptimo de perfiles por grupo (AISC 360-16 ASD + NCh2369:2025)

**Entradas que debes subir a Kaggle (dataset privado):**
1. El modelo SAP2000 con resultados (`.s2k` o `.txt`, el mismo que me pasaste comprimido).
2. `base_datos_perfiles.csv` (base de datos de perfiles validada).
3. (Opcional) `parametros_grupos.csv`: los parámetros por grupo que quieras fijar fuera del notebook. Si no lo subes, se usan los de la celda de configuración.

**Qué hace:** lee las barras de cada grupo y sus demandas ASD (32 combinaciones, todas las estaciones), evalúa cada perfil de la base de datos y entrega, por grupo, los perfiles que cumplen resistencia (compresión, tracción, flexión con pandeo lateral-torsional, corte e interacción H1), esbeltez local (NCh2369:2025 Tabla 9, con los vacíos completados desde la 2003) y esbeltez global (KL/r). Los ordena por peso total del grupo.

**Parámetros editables por grupo** (celda de configuración): tipo de miembro, familias permitidas, Kx, Ky, factor de longitud sin arriostrar, Cb, factor de utilización máximo, límite KL/r y excepción de 8.8.4 (0,7·R1 ≥ 1,0).

**Pendiente de confirmar antes de usar resultados:** Fy, Ry y E (valores de prueba en la celda de configuración), la aplicación de la excepción 8.8.4 por grupo y el criterio de Ry·Fy frente a Fy en la Tabla 9.


In [ ]:
import math, re, glob, os, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd

# ---- Materiales (ingresar) ----
FY_MPA = 248.2      # Fy [MPa]. Modelo: A36 = 2531 kgf/cm2 (= 248.2 MPa)  <-- CONFIRMAR
RY = 1.5            # Ry (factor de fluencia esperada)                     <-- CONFIRMAR
E_MPA = 200000.0    # E [MPa]
USE_RY = True       # True: Ry·Fy en Tabla 9 (conservador)
# ---- Diseño ----
DESIGN = 'ASD'      # el .s2k del modelo está en ASD (AISC 360-16)
UTIL_MAX = 1.0      # factor de utilización máximo por defecto (se puede cambiar por grupo)
LIM_KLR_DEFAULT = 200.0
# ---- Rutas ----
INPUT_DIR = '/kaggle/input'
WORK_DIR = '/kaggle/working'
MODEL_FILE = None           # None = detectar .s2k o .txt con tablas SAP
BASE_DB = None              # None = buscar base_datos_perfiles.csv
# ---- Parámetros por grupo (editables). Una fila por grupo del modelo ----
GRUPOS = pd.DataFrame([
    # grupo, tipo, familias permitidas, Kx, Ky, Lb_factor, Cb, util_max, lim_KLr
    ['COLUMNAS', 'Columna', 'I soldada,I/H ala ancha,I,H,HSS cuadrado,HSS rectangular', 1.0, 1.0, 1.0, 1.0, UTIL_MAX, LIM_KLR_DEFAULT, 'No'],
    ['VIGAS MARCOS', 'Viga', 'I soldada,I/H ala ancha,I,H', 1.0, 1.0, 1.0, 1.0, UTIL_MAX, LIM_KLR_DEFAULT, 'No'],
    ['ARRIOSTRAMIENTO PISO', 'Arriostramiento', 'L (ángulo),HSS cuadrado,HSS rectangular,HSS circular', 1.0, 1.0, 1.0, 1.0, UTIL_MAX, None, 'No'],
    ['ARRIOSTRAMIENTOS AX', 'Arriostramiento', 'L (ángulo),HSS cuadrado,HSS rectangular,HSS circular', 1.0, 1.0, 1.0, 1.0, UTIL_MAX, None, 'No'],
    ['ARRIOSTRAMIENTOS AY', 'Arriostramiento', 'L (ángulo),HSS cuadrado,HSS rectangular,HSS circular', 1.0, 1.0, 1.0, 1.0, UTIL_MAX, None, 'No'],
], columns=['grupo', 'tipo_miembro', 'familias', 'Kx', 'Ky', 'Lb_factor', 'Cb', 'util_max', 'lim_KLr', 'excepcion_0_7R1'])
# Si existe parametros_grupos.csv en la entrada (editado y subido a Kaggle), reemplaza la tabla de arriba
_csv_grupos = glob.glob(f'{INPUT_DIR}/**/parametros_grupos.csv', recursive=True)
if _csv_grupos:
    GRUPOS = pd.read_csv(_csv_grupos[0])
    print('Parámetros de grupos leídos desde', _csv_grupos[0])
GRUPOS.to_csv(os.path.join(WORK_DIR, 'parametros_grupos_plantilla.csv'), index=False) if os.path.isdir(WORK_DIR) else None
print('Fy =', FY_MPA, 'MPa | Ry =', RY, '| RyFy =', round(RY * FY_MPA, 1), 'MPa | E =', E_MPA, 'MPa')

In [ ]:
KV = re.compile(r'(\w+)=("[^"]*"|\S+)')
def find_model():
    if MODEL_FILE: return MODEL_FILE
    hits = [p for p in glob.glob(f'{INPUT_DIR}/**/*', recursive=True) if p.lower().endswith(('.s2k', '.txt'))]
    hits = [p for p in hits if open(p, encoding='latin-1', errors='replace').read(4000).find('TABLE:') >= 0]
    if len(hits) != 1:
        raise FileNotFoundError(f'Se esperaba un modelo .s2k/.txt en {INPUT_DIR}; encontrados: {hits}. Fija MODEL_FILE.')
    return hits[0]

def logical_lines(path):
    """Une las líneas que SAP parte con ' _' al final."""
    buf = ''
    with open(path, encoding='latin-1', errors='replace') as f:
        for line in f:
            s = line.rstrip('\n')
            if s.rstrip().endswith('_'):
                buf += s.rstrip()[:-1] + ' '
                continue
            yield buf + s
            buf = ''

WANT = {'JOINT COORDINATES', 'JOINT DISPLACEMENTS', 'PROGRAM CONTROL', 'PREFERENCES - STEEL DESIGN - AISC 360-16', 'CONNECTIVITY - FRAME',
        'FRAME SECTION ASSIGNMENTS', 'GROUPS 2 - ASSIGNMENTS', 'ELEMENT FORCES - FRAMES'}
def read_model(path, grupos):
    out = {k: [] for k in WANT}
    cur = None
    gset = set(grupos)
    frames_in_groups = set()
    for ln in logical_lines(path):
        s = ln.strip()
        if s.startswith('TABLE:'):
            m = re.match(r'TABLE:\s+"(.+?)"', s)
            cur = m.group(1) if m else None
            continue
        if cur not in WANT or '=' not in s:
            continue
        if cur == 'GROUPS 2 - ASSIGNMENTS':
            d = {k: v.strip('"') for k, v in KV.findall(s)}
            if d.get('ObjectType') == 'Frame' and d.get('GroupName') in gset:
                out[cur].append((d['GroupName'], d['ObjectLabel']))
        elif cur == 'ELEMENT FORCES - FRAMES':
            if 'OutputCase="ASD' not in s and 'OutputCase=ASD' not in s:
                continue
            d = {k: v.strip('"') for k, v in KV.findall(s)}
            out[cur].append((d['Frame'], float(d['Station']), d['OutputCase'], float(d['P']), float(d['V2']),
                             float(d['V3']), float(d['M2']), float(d['M3'])))
        elif cur == 'JOINT COORDINATES':
            d = {k: v.strip('"') for k, v in KV.findall(s)}
            out[cur].append((d['Joint'], float(d['GlobalX']), float(d['GlobalY']), float(d['GlobalZ'])))
        elif cur == 'JOINT DISPLACEMENTS':
            if 'OutputCase="ASD' not in s and 'OutputCase=ASD' not in s:
                continue
            d = {k: v.strip('"') for k, v in KV.findall(s)}
            out[cur].append((d['Joint'], d['OutputCase'], float(d['U1']), float(d['U2']), float(d['U3'])))
        elif cur == 'PREFERENCES - STEEL DESIGN - AISC 360-16':
            d = {k: v.strip('"') for k, v in KV.findall(s)}
            if 'OmegaB' in d: out[cur].append(d)
        elif cur == 'PROGRAM CONTROL':
            d = {k: v.strip('"') for k, v in KV.findall(s)}
            if 'CurrUnits' in d: out[cur].append(d)
        else:
            d = {k: v.strip('"') for k, v in KV.findall(s)}
            out[cur].append(d)
    return out

model_path = find_model()
print('Modelo:', model_path, '|', round(os.path.getsize(model_path) / 1e6, 1), 'MB')
raw = read_model(model_path, GRUPOS['grupo'].tolist())
units = raw['PROGRAM CONTROL'][0]['CurrUnits'] if raw['PROGRAM CONTROL'] else '?'
print('Unidades del modelo:', units)
assert [p.strip().lower() for p in units.split(',')][:2] == ['kgf', 'm'], 'El modelo debe estar en kgf-m'
pref = raw['PREFERENCES - STEEL DESIGN - AISC 360-16'][0]
OMEGA_C = float(pref.get('OmegaC', 1.67)); OMEGA_B = float(pref.get('OmegaB', 1.67))
OMEGA_T = float(pref.get('OmegaTY', 1.67)); OMEGA_V = float(pref.get('OmegaV', 1.67))
OMEGA_V_ROLLED = float(pref.get('OmegaVRolledI', 1.5))
print(f'Ω del modelo: Ωc={OMEGA_C} Ωb={OMEGA_B} Ωt={OMEGA_T} Ωv={OMEGA_V} Ωv(I laminado)={OMEGA_V_ROLLED}')

In [ ]:
KGF = 9.80665
conn = pd.DataFrame(raw['CONNECTIVITY - FRAME'])[['Frame', 'Length']]
conn['L_mm'] = conn['Length'].astype(float) * 1000
sap_sec = pd.DataFrame(raw['FRAME SECTION ASSIGNMENTS'])[['Frame', 'AnalSect']]
fr = pd.DataFrame(raw['GROUPS 2 - ASSIGNMENTS'], columns=['grupo', 'Frame'])
fr = fr.merge(conn, on='Frame', how='left').merge(sap_sec, on='Frame', how='left')
fr = fr.merge(GRUPOS[['grupo', 'tipo_miembro', 'Kx', 'Ky', 'Lb_factor', 'Cb', 'util_max', 'lim_KLr', 'excepcion_0_7R1']], on='grupo')
missing = fr[fr['L_mm'].isna()]
assert missing.empty, f'Barras de grupo sin conectividad: {missing.Frame.tolist()}'
print('Barras en grupos:', len(fr), '| por grupo:', fr.groupby('grupo').size().to_dict())

dem = pd.DataFrame(raw['ELEMENT FORCES - FRAMES'], columns=['Frame', 'Station', 'combo', 'P', 'V2', 'V3', 'M2', 'M3'])
dem = dem[dem['Frame'].isin(fr['Frame'])].copy()
dem['P'] *= KGF; dem['V2'] *= KGF; dem['V3'] *= KGF
dem['M2'] *= KGF * 1000; dem['M3'] *= KGF * 1000          # kgf·m -> N·mm
dem = dem.merge(fr[['Frame', 'grupo']], on='Frame')
print('Filas de demanda ASD:', len(dem), '| combinaciones:', dem['combo'].nunique())
print(dem.groupby('grupo').agg(Pc_max_kN=('P', lambda s: round(-s.min()/1000, 1)),
                              Pt_max_kN=('P', lambda s: round(s.max()/1000, 1)),
                              M3_max_kNm=('M3', lambda s: round(s.abs().max()/1e6, 2)),
                              M2_max_kNm=('M2', lambda s: round(s.abs().max()/1e6, 2))))

In [ ]:
import io
# Base de datos de perfiles incrustada (versión validada). Si subes base_datos_perfiles.csv a la entrada, se usa ese archivo.
DB_EMBEBIDA = r'''familia,tipo,designacion,H_mm,B_mm,tw_mm,tf_mm,A_cm2,peso_kgf_m,Ix,Iy,Wx,Wy,Zx,Zy,ix,iy,J,Iyc,ro_cm,ia,it,fuente,validado,nota,r_min_cm
Copromet IPE,I,IPE 80,80.0,46.0,3.8,5.2,7.6,6.0,80.1,8.4,20.0,3.6,22.5,5.8,3.24,1.05,0.56,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPE,I,IPE 100,100.0,55.0,4.1,5.7,10.3,8.1,171.0,15.9,34.2,5.7,37.6,9.0,4.07,1.24,0.88,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPE,I,IPE 120,120.0,64.0,4.4,6.3,13.2,10.4,318.0,27.7,53.0,8.65,58.5,13.4,4.9,1.45,1.37,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPE,I,IPE 140,140.0,73.0,4.7,6.9,16.4,12.9,541.0,44.9,77.3,12.3,85.8,19.1,5.74,1.65,2.04,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPE,I,IPE 160,160.0,82.0,5.0,7.4,20.0,15.8,869.0,68.3,109.0,16.7,119.0,25.8,6.58,1.84,2.82,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPE,I,IPE 180,180.0,91.0,5.3,8.0,23.9,18.8,1320.0,101.0,146.0,22.2,160.9,34.3,7.43,2.05,3.92,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPE,I,IPE 200,200.0,100.0,5.6,8.5,28.5,22.4,1940.0,142.0,194.0,28.5,209.7,43.9,8.25,2.24,5.17,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPE,I,IPE 220,220.0,110.0,5.9,9.2,33.4,26.2,2770.0,205.0,252.0,37.3,273.3,57.4,9.11,2.48,7.09,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPE,I,IPE 240,240.0,120.0,6.2,9.8,39.0,30.7,3890.0,284.0,324.0,47.3,346.0,72.7,9.97,2.7,9.28,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPE,I,IPE 270,270.0,135.0,6.6,10.2,45.9,36.1,5790.0,420.0,429.0,62.2,460.5,95.7,11.23,3.02,11.94,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPE,I,IPE 300,300.0,150.0,7.1,10.7,53.8,42.2,8360.0,604.0,557.0,80.5,602.1,123.9,12.47,3.35,15.57,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPE,I,IPE 330,330.0,160.0,7.5,11.5,62.6,49.1,11770.0,788.0,713.0,98.5,762.8,151.5,13.7,3.55,20.54,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPE,I,IPE 360,360.0,170.0,8.0,12.7,72.7,57.1,16270.0,1040.0,904.0,123.0,973.7,188.9,14.96,3.78,28.93,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPE,I,IPE 400,400.0,180.0,8.6,13.5,84.5,66.3,23130.0,1320.0,1160.0,146.0,1238.3,225.6,16.54,3.95,37.43,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPE,I,IPE 450,450.0,190.0,9.4,14.6,98.8,77.7,33740.0,1680.0,1500.0,176.0,1623.9,272.8,18.48,4.12,51.07,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPE,I,IPE 500,500.0,200.0,10.2,16.0,116.0,90.7,48200.0,2140.0,1930.0,214.0,2107.3,332.2,20.38,4.3,71.17,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPE,I,IPE 550,550.0,210.0,11.1,17.2,134.0,106.0,67120.0,2670.0,2440.0,254.0,2662.2,395.1,22.38,4.77,94.74,,,,,Catálogo Copromet 2024,revisar,Iy no cuadra con iy Zx/Zy/J aprox. desde geometría,
Copromet IPE,I,IPE 600,600.0,220.0,12.0,19.0,156.0,122.0,92080.0,3387.0,3070.0,308.0,3376.1,480.0,24.3,4.66,132.97,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPN,I inclinada,IPN 80,80.0,42.0,3.9,5.9,7.58,5.95,77.8,6.29,19.5,3.0,22.9,5.5,3.2,0.91,0.71,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPN,I inclinada,IPN 100,100.0,50.0,4.5,6.8,10.6,8.32,171.0,12.2,34.2,4.88,40.1,8.9,4.01,1.07,1.31,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPN,I inclinada,IPN 120,120.0,58.0,5.1,7.7,14.2,11.2,328.0,21.5,54.7,7.41,64.1,13.6,4.81,1.23,2.23,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPN,I inclinada,IPN 140,140.0,66.0,5.7,8.6,18.3,14.4,573.0,35.2,81.9,10.7,96.1,19.7,5.61,1.4,3.56,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPN,I inclinada,IPN 160,160.0,74.0,6.3,9.5,22.8,17.9,935.0,54.7,117.0,14.8,137.1,27.4,6.4,1.55,5.4,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPN,I inclinada,IPN 180,180.0,82.0,6.9,10.4,27.9,21.9,1450.0,81.3,161.0,19.8,188.4,36.9,7.2,1.71,7.89,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPN,I inclinada,IPN 200,200.0,90.0,7.5,11.3,33.5,26.3,2140.0,117.0,214.0,26.0,250.9,48.3,8.0,1.87,11.15,,,,,Catálogo Copromet 2024,revisar,Ix corregido x10 (tabla 214.0) Zx/Zy/J aprox. desde geometría,
Copromet IPN,I inclinada,IPN 220,220.0,98.0,8.1,12.2,39.6,31.1,3060.0,162.0,278.0,33.1,325.9,61.8,8.8,2.02,15.33,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPN,I inclinada,IPN 240,240.0,106.0,8.7,13.1,46.1,36.2,4250.0,221.0,354.0,41.7,414.5,77.6,9.59,2.2,20.58,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPN,I inclinada,IPN 260,260.0,113.0,9.4,14.1,53.4,41.9,5740.0,288.0,442.0,51.0,518.1,95.1,10.4,2.32,27.54,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPN,I inclinada,IPN 300,300.0,125.0,10.8,16.2,69.1,54.2,9800.0,451.0,653.0,72.2,768.0,134.4,11.9,2.56,46.67,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPN,I inclinada,IPN 340,340.0,137.0,12.2,18.3,86.8,68.1,15700.0,674.0,923.0,98.4,1087.3,183.0,13.5,2.8,74.34,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPN,I inclinada,IPN 360,360.0,143.0,13.0,19.5,97.1,76.2,19610.0,818.0,1090.0,114.0,1284.4,212.9,14.2,2.9,94.2,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPN,I inclinada,IPN 400,400.0,155.0,14.4,21.6,118.0,92.6,29210.0,1160.0,1460.0,149.0,1725.2,278.0,15.7,3.13,139.65,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPN,I inclinada,IPN 450,450.0,170.0,16.2,24.3,147.0,115.0,45850.0,1730.0,2040.0,203.0,2411.1,377.5,17.7,3.43,219.51,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPN,I inclinada,IPN 500,500.0,185.0,18.0,27.0,180.0,141.0,68740.0,2480.0,2750.0,268.0,3257.8,498.2,19.6,3.72,329.46,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet IPN,I inclinada,IPN 550,550.0,200.0,19.0,30.0,213.0,167.0,99180.0,3490.0,3610.0,349.0,4260.5,644.2,21.6,4.02,472.03,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEA,H,HEA 100,96.0,100.0,5.0,8.0,21.2,16.7,349.0,134.0,72.8,26.8,78.4,40.5,4.06,2.51,3.75,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEA,H,HEA 120,114.0,120.0,5.0,8.0,25.3,19.9,606.0,231.0,106.0,38.5,113.8,58.2,4.89,3.01,4.5,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEA,H,HEA 140,133.0,140.0,5.5,8.5,31.4,24.7,1030.0,389.0,155.0,55.6,166.7,84.2,5.73,3.52,6.38,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEA,H,HEA 160,152.0,160.0,6.0,9.0,38.8,30.4,1670.0,616.0,220.0,76.9,232.9,116.4,6.57,3.98,8.74,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEA,H,HEA 180,171.0,180.0,6.0,9.5,45.3,35.5,2510.0,925.0,294.0,103.0,310.8,155.3,7.45,4.52,11.38,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEA,H,HEA 200,190.0,200.0,6.5,10.0,53.8,42.3,3690.0,1340.0,389.0,134.0,407.0,201.8,8.28,4.98,14.89,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEA,H,HEA 220,210.0,220.0,7.0,11.0,64.3,50.5,5410.0,1950.0,515.0,178.0,543.4,268.5,9.17,5.51,21.67,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEA,H,HEA 240,230.0,240.0,7.5,12.0,76.8,60.3,7760.0,2770.0,675.0,231.0,707.4,348.5,10.1,6.0,30.54,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEA,H,HEA 260,250.0,260.0,7.5,12.5,86.8,68.2,10450.0,3670.0,836.0,282.0,866.8,425.7,11.0,6.5,37.02,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEA,H,HEA 280,270.0,280.0,8.0,13.0,97.3,76.4,13670.0,4760.0,1010.0,340.0,1054.6,513.5,11.9,7.0,45.17,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEA,H,HEA 300,290.0,300.0,8.5,14.0,112.0,88.3,18260.0,6310.0,1260.0,421.0,1305.1,634.7,12.8,7.5,60.24,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEA,H,HEA 320,310.0,300.0,9.0,15.5,124.4,97.6,22928.0,6985.0,1480.0,466.0,1544.6,703.1,13.6,7.49,81.26,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEA,H,HEA 340,330.0,300.0,9.5,16.5,133.5,105.0,27693.0,7436.0,1680.0,496.0,1761.3,749.2,14.4,7.46,98.33,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEA,H,HEA 360,350.0,300.0,10.0,17.5,142.8,112.0,33090.0,7887.0,1890.0,526.0,1993.7,795.4,15.2,7.43,117.69,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEA,H,HEA 400,390.0,300.0,11.0,19.0,159.0,125.0,45069.0,8564.0,2310.0,571.0,2455.4,865.6,16.8,7.34,152.8,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEA,H,HEA 450,440.0,300.0,11.5,21.0,178.0,140.0,63722.0,9465.0,2900.0,631.0,3095.1,958.2,18.9,7.29,205.4,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEA,H,HEA 500,490.0,300.0,12.0,23.0,197.5,155.0,86975.0,10367.0,3550.0,691.0,3813.7,1051.0,21.0,17.24,268.91,,,,,Catálogo Copromet 2024,revisar,Iy no cuadra con iy Zx/Zy/J aprox. desde geometría,
Copromet HEB,H,HEB 100,100.0,100.0,6.0,10.0,26.0,20.4,450.0,167.0,89.0,33.5,99.6,50.7,4.16,2.53,7.24,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEB,H,HEB 120,120.0,120.0,6.5,11.0,34.0,26.7,864.0,318.0,144.0,52.9,159.5,80.2,5.04,3.06,11.55,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEB,H,HEB 140,140.0,140.0,7.0,12.0,43.0,33.7,1510.0,550.0,216.0,78.5,238.6,119.0,5.93,3.58,17.45,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEB,H,HEB 160,160.0,160.0,8.0,13.0,54.3,42.6,2490.0,889.0,311.0,111.0,341.7,168.5,6.77,4.05,25.72,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEB,H,HEB 180,180.0,180.0,8.5,14.0,65.3,51.2,3830.0,1360.0,426.0,151.0,467.4,229.5,7.66,4.56,36.04,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEB,H,HEB 200,200.0,200.0,9.0,15.0,78.1,61.3,5700.0,2000.0,570.0,200.0,620.0,303.4,8.54,5.06,49.13,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEB,H,HEB 220,220.0,220.0,9.5,16.0,91.0,71.5,8090.0,2840.0,736.0,258.0,802.0,391.4,9.43,5.59,65.45,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEB,H,HEB 240,240.0,240.0,10.0,17.0,106.0,83.2,11260.0,3920.0,938.0,327.0,1015.9,494.8,10.31,6.08,85.47,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEB,H,HEB 260,260.0,260.0,10.0,17.5,118.0,93.0,14920.0,5130.0,1150.0,395.0,1229.9,597.1,11.24,6.59,100.4,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEB,H,HEB 280,280.0,280.0,10.5,18.0,131.0,103.0,19270.0,6590.0,1380.0,471.0,1476.8,712.3,12.12,7.09,118.28,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEB,H,HEB 300,300.0,300.0,11.0,19.0,149.0,117.0,25170.0,8560.0,1680.0,571.0,1790.5,862.9,13.0,7.58,148.8,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEB,H,HEB 320,320.0,300.0,11.5,20.5,161.3,127.0,30820.0,9239.0,1930.0,616.0,2065.7,931.7,13.84,7.58,186.45,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEB,H,HEB 340,340.0,300.0,12.0,21.5,170.9,134.0,36656.0,9690.0,2160.0,646.0,2319.0,978.2,14.6,7.53,215.87,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEB,H,HEB 360,360.0,300.0,12.5,22.5,180.6,142.0,43193.0,10141.0,2400.0,676.0,2588.2,1024.8,15.5,7.49,248.32,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEB,H,HEB 400,400.0,300.0,13.5,24.0,197.8,155.0,57680.0,10819.0,2880.0,721.0,3125.4,1096.0,17.1,7.4,305.35,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEB,H,HEB 450,450.0,300.0,14.5,26.0,218.0,171.0,79887.0,11721.0,3550.0,781.0,3881.4,1190.9,19.1,7.33,391.97,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet HEB,H,HEB 500,500.0,300.0,14.5,28.0,238.6,187.0,107176.0,12624.0,4290.0,842.0,4679.4,1283.3,21.2,7.27,484.16,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet UPN,U,UPN 50,50.0,38.0,5.0,7.0,7.12,5.59,26.4,9.12,10.6,3.75,,,1.93,1.13,,,,,,Catálogo Copromet 2024,sí,,
Copromet UPN,U,UPN 65,65.0,42.0,5.5,7.5,9.03,7.09,57.5,14.1,17.6,5.07,,,2.52,1.25,,,,,,Catálogo Copromet 2024,sí,,
Copromet UPN,U,UPN 80,80.0,45.0,6.0,8.0,11.0,8.64,106.0,19.4,26.5,6.36,,,3.1,1.33,,,,,,Catálogo Copromet 2024,sí,,
Copromet UPN,U,UPN 100,100.0,50.0,6.0,8.5,13.5,10.6,206.0,29.3,41.2,8.49,,,3.91,1.47,,,,,,Catálogo Copromet 2024,sí,,
Copromet UPN,U,UPN 120,120.0,55.0,7.0,9.0,17.0,13.4,364.0,43.2,60.7,11.1,,,4.62,1.59,,,,,,Catálogo Copromet 2024,sí,,
Copromet UPN,U,UPN 140,140.0,60.0,7.0,10.0,20.4,16.0,605.0,62.7,86.4,14.8,,,5.45,1.75,,,,,,Catálogo Copromet 2024,sí,,
Copromet UPN,U,UPN 160,160.0,65.0,7.5,10.5,24.0,18.8,925.0,85.3,116.0,18.3,,,6.21,1.89,,,,,,Catálogo Copromet 2024,sí,,
Copromet UPN,U,UPN 180,180.0,70.0,8.0,11.0,28.0,22.0,1350.0,114.0,150.0,22.4,,,6.95,2.02,,,,,,Catálogo Copromet 2024,sí,,
Copromet UPN,U,UPN 200,200.0,75.0,8.5,11.5,32.2,25.3,1910.0,148.0,191.0,27.0,,,7.7,2.14,,,,,,Catálogo Copromet 2024,sí,,
Copromet UPN,U,UPN 220,220.0,80.0,9.0,12.5,37.4,29.4,2690.0,197.0,245.0,33.6,,,8.48,2.3,,,,,,Catálogo Copromet 2024,sí,,
Copromet UPN,U,UPN 240,240.0,85.0,9.5,13.0,42.3,33.2,3600.0,248.0,300.0,39.6,,,9.22,2.42,,,,,,Catálogo Copromet 2024,sí,,
Copromet UPN,U,UPN 260,260.0,90.0,10.0,14.0,48.3,37.9,4820.0,317.0,371.0,47.7,,,9.99,2.56,,,,,,Catálogo Copromet 2024,sí,,
Copromet UPN,U,UPN 280,280.0,95.0,10.0,15.0,53.3,41.8,6280.0,399.0,448.0,57.2,,,10.9,2.74,,,,,,Catálogo Copromet 2024,sí,,
Copromet UPN,U,UPN 300,300.0,100.0,10.0,16.0,58.8,46.2,8030.0,495.0,535.0,67.8,,,11.69,2.9,,,,,,Catálogo Copromet 2024,sí,,
Copromet UPN,U,UPN 320,320.0,100.0,14.0,17.5,75.9,59.5,10870.0,597.0,679.0,80.6,,,12.1,2.81,,,,,,Catálogo Copromet 2024,sí,,
Copromet UPN,U,UPN 350,350.0,100.0,14.0,16.0,77.3,60.6,12840.0,570.0,734.0,75.0,,,12.9,2.72,,,,,,Catálogo Copromet 2024,sí,,
Copromet UPN,U,UPN 380,380.0,102.0,13.5,16.0,80.4,63.1,15760.0,615.0,828.0,78.7,,,14.0,2.77,,,,,,Catálogo Copromet 2024,sí,,
Copromet UPN,U,UPN 400,400.0,110.0,14.0,18.0,91.5,71.8,20350.0,846.0,1020.0,102.0,,,14.9,3.04,,,,,,Catálogo Copromet 2024,sí,,
Copromet W (ASTM A6),I/H ala ancha,"W 150 x 13,5",150.0,100.0,4.3,5.5,17.29,13.5,685.5,91.8,91.4,18.36,100.2,28.1,6.29,2.3,1.48,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 150 x 18,0",153.0,102.0,5.8,7.1,22.84,18.0,915.9,125.9,122.1,25.37,133.6,38.1,6.33,2.36,3.34,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 150 x 22,5",152.0,152.0,5.8,6.6,28.42,22.5,1206.0,387.0,158.6,50.9,173.8,77.4,6.51,3.65,3.82,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 150 x 29,8",157.0,153.0,6.6,9.3,38.5,29.8,1714.0,556.0,221.5,72.6,241.8,110.4,6.72,3.8,9.53,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 150 x 37,1",162.0,154.0,8.1,11.6,47.28,37.1,2244.0,707.0,277.0,91.8,307.7,139.8,6.85,3.84,18.48,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 200 x 15,0",200.0,100.0,4.3,5.2,19.1,15.0,1280.0,86.89,128.0,17.38,139.9,26.9,8.18,2.3,1.44,,,,,Catálogo Copromet 2024,revisar,Iy no cuadra con iy Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 200 x 19,3",203.0,102.0,5.8,6.5,24.83,19.3,1662.0,116.0,166.1,22.7,182.6,35.4,8.19,2.14,3.1,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 200 x 22,5",206.0,102.0,6.2,8.0,29.0,22.5,2004.0,142.0,197.0,27.9,217.5,43.4,8.37,2.22,4.99,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 200 x 26,6",207.0,133.0,5.8,8.4,33.92,26.6,2611.0,330.0,252.3,49.6,274.3,75.9,8.73,3.1,6.49,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 200 x 31,3",210.0,134.0,6.4,10.2,40.3,31.3,3168.0,410.0,301.7,61.2,330.6,93.5,8.86,3.19,11.14,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 200 x 35,9",201.0,165.0,6.2,10.2,45.7,35.9,3450.0,762.0,342.0,92.3,371.7,140.6,8.69,4.09,13.11,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 200 x 41,7",205.0,166.0,7.2,11.8,53.2,41.7,4080.0,903.0,398.0,109.0,437.7,164.9,8.76,4.11,20.44,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 200 x 46,1",203.0,203.0,7.2,11.0,58.6,46.1,4545.0,1535.0,451.0,151.2,487.7,229.0,8.81,5.3,20.26,,,,,Catálogo Copromet 2024,revisar,Iy no cuadra con iy Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 200 x 52,0",206.0,204.0,7.9,12.7,66.5,52.0,5268.0,1784.0,511.0,174.0,565.2,267.1,8.92,5.16,30.83,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 200 x 71,0",216.0,206.0,10.2,17.4,91.0,71.0,7660.0,2537.0,709.2,246.3,795.6,373.9,9.17,5.28,78.76,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 200 x 86,0",222.0,209.0,12.9,20.6,110.5,86.0,9467.0,3130.0,852.0,300.0,972.5,457.4,9.27,5.33,134.74,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 250 x 17,9",251.0,101.0,4.8,5.3,22.79,17.9,2252.0,91.34,179.5,18.09,200.9,28.4,9.92,2.0,1.89,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 250 x 22,3",254.0,102.0,5.8,6.9,28.56,22.3,2901.0,122.6,231.4,24.1,257.6,37.9,10.09,2.06,3.8,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 250 x 25,3",257.0,102.0,6.1,8.4,32.34,25.3,3430.0,149.0,270.2,29.3,301.0,45.9,10.31,2.14,5.85,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 250 x 28,4",260.0,102.0,6.4,10.0,36.31,28.4,3998.0,178.0,311.2,34.8,347.2,54.5,10.51,2.2,8.9,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 250 x 32,7",258.0,146.0,6.1,9.1,41.75,32.7,4895.0,473.0,382.7,64.8,418.4,99.2,10.83,3.35,9.15,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 250 x 38,5",262.0,147.0,6.6,11.2,49.29,38.5,6057.0,594.0,462.4,80.8,507.6,123.6,11.05,3.46,16.06,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 250 x 44,8",266.0,148.0,7.6,13.0,57.6,44.8,7118.0,704.0,538.2,95.1,596.2,145.8,11.15,3.5,25.19,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 250 x 58,0",252.0,203.0,8.0,13.5,74.2,58.0,8700.0,1870.0,690.0,185.0,754.9,281.8,10.8,5.03,37.14,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 250 x 67,0",257.0,204.0,8.9,15.8,85.8,67.0,10300.0,2220.0,805.0,218.0,890.5,333.2,11.0,5.11,58.94,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 250 x 73,0",253.0,254.0,8.6,14.2,92.9,73.0,11290.0,3880.0,889.0,305.5,969.8,462.2,11.02,6.47,53.25,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 250 x 80,0",256.0,255.0,9.4,15.6,102.1,80.0,12570.0,4314.0,980.5,338.3,1075.1,512.2,11.1,6.51,70.76,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 250 x 89,0",260.0,256.0,10.7,17.3,113.9,89.0,14237.0,4841.0,1095.1,378.2,1210.8,573.3,11.18,6.52,97.57,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 250 x 101,0",264.0,257.0,11.9,19.6,129.0,101.0,16400.0,5580.0,1240.0,433.0,1381.4,655.2,11.3,6.58,141.63,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 310 x 23,8",305.0,101.0,5.6,6.7,30.7,23.8,4280.0,113.0,285.0,22.9,320.9,36.5,11.89,1.94,3.73,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 310 x 28,3",309.0,102.0,6.0,8.9,36.5,28.3,5431.0,158.0,356.0,31.0,399.6,48.9,12.28,2.08,6.89,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 310 x 32,7",313.0,102.0,6.6,10.8,41.81,32.7,6570.0,192.0,419.8,37.6,473.0,59.4,12.49,2.13,11.36,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 310 x 38,7",310.0,165.0,5.8,9.7,49.41,38.7,8527.0,726.0,553.6,88.1,603.1,134.5,13.14,3.82,11.93,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 310 x 44,5",313.0,166.0,6.6,11.2,56.91,44.5,9934.0,855.0,638.8,103.0,700.4,157.5,13.22,3.87,18.33,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 310 x 52,0",317.0,167.0,7.6,13.2,67.0,52.0,11850.0,1026.0,751.4,122.9,830.1,188.3,13.33,3.91,29.86,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 310 x 60,0",303.0,203.0,7.5,13.1,75.88,60.0,12860.0,1829.0,850.0,180.0,914.6,273.8,13.0,4.9,34.32,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 310 x 74,0",310.0,205.0,9.4,16.3,94.84,74.0,16450.0,2340.0,1060.0,228.0,1162.2,348.6,13.2,4.98,66.87,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 310 x 79,0",306.0,254.0,8.8,14.6,100.5,79.0,17670.0,3990.0,1155.0,314.2,1249.2,476.3,13.26,6.3,58.99,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 310 x 97,0",308.0,305.0,9.9,15.4,123.6,97.0,22240.0,7286.0,1444.0,477.8,1564.5,723.1,13.43,7.68,83.23,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 310 x 107,0",311.0,306.0,10.9,17.0,136.4,107.0,24790.0,8123.0,1597.3,530.9,1738.5,804.1,13.49,7.72,112.18,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 310 x 117,0",314.0,307.0,11.9,18.7,149.9,117.0,27510.0,9024.0,1756.6,587.9,1922.9,891.0,13.56,7.76,149.37,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 310 x 129,0",318.0,308.0,13.1,20.5,165.2,129.0,30770.0,10004.0,1930.0,651.0,2129.7,984.2,13.7,7.8,197.65,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 310 x 32,9",349.0,127.0,5.8,8.5,41.7,32.9,8258.0,291.0,479.0,45.9,527.4,71.3,14.09,2.63,7.36,,,,,Catálogo Copromet 2024,revisar,Ix corregido x100 (tabla 82.58) Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 360 x 39,0",353.0,128.0,6.5,10.7,49.8,39.0,10230.0,375.0,585.3,58.6,647.5,91.2,14.35,2.73,13.49,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 360 x 44,6",352.0,171.0,6.9,9.8,57.31,44.6,12258.0,818.0,696.5,95.7,764.1,147.2,14.58,3.77,14.37,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 360 x 51,0",355.0,171.0,7.2,11.6,64.8,51.0,14130.0,968.0,801.2,113.3,879.3,173.9,14.81,3.87,21.92,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 360 x 57,8",358.0,172.0,7.9,13.1,72.13,57.8,16143.0,1113.0,901.8,129.4,994.6,199.0,14.92,3.92,31.23,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 360 x 64,0",347.0,203.0,7.7,13.5,81.7,64.0,17890.0,1885.0,1027.0,185.7,1111.1,282.9,14.8,4.8,38.17,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 360 x 72,0",350.0,204.0,8.6,15.1,91.04,72.0,20169.0,2140.0,1152.5,209.8,1251.5,320.1,14.86,4.84,53.6,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 360 x 79,0",354.0,205.0,9.4,16.8,101.2,79.0,22713.0,2416.0,1283.2,235.7,1402.6,360.1,14.98,4.89,73.67,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 360 x 91,0",353.0,254.0,9.5,16.4,115.9,91.0,26690.0,4450.0,1510.0,352.0,1645.6,536.3,15.2,6.22,83.84,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 360 x 101,0",357.0,255.0,20.5,18.3,129.5,101.0,30100.0,5040.0,1690.0,397.0,2106.7,628.6,15.3,6.25,196.19,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 360 x 110,0",360.0,256.0,11.4,19.9,140.6,110.0,35100.0,5580.0,1840.0,436.0,2024.8,662.5,15.3,6.3,150.31,,,,,Catálogo Copromet 2024,revisar,ix de tabla (15.3) no cuadra con sqrt(Ix/A)=15.8; revisar Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 360 x 122,0",363.0,257.0,13.0,21.7,153.3,122.0,36530.0,6160.0,201.3,480.0,2235.4,730.1,15.4,6.3,198.48,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 360 x 134,0",356.0,369.0,11.12,18.0,171.0,134.0,41600.0,15100.0,2340.0,818.0,2529.7,1235.3,15.6,9.4,158.13,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 360 x 147,0",360.0,370.0,12.3,19.8,187.7,147.0,46200.0,16700.0,2570.0,905.0,2808.0,1367.4,15.7,9.42,211.35,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 360 x 162,0",364.0,371.0,13.3,21.8,206.5,162.0,51600.0,18600.0,2830.0,1000.0,3109.0,1514.5,15.8,9.47,281.37,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 360 x 179,0",368.0,373.0,15.0,23.9,227.7,179.0,57400.0,20600.0,3110.0,1110.0,3452.0,1680.6,15.8,9.5,375.5,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 360 x 196,0",372.0,374.0,16.4,26.2,250.3,196.0,63700.0,22800.0,3420.0,1220.0,3807.2,1853.9,16.0,9.55,495.41,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 410 x 38,8",399.0,140.0,6.4,8.8,50.3,38.8,12620.0,404.0,632.6,57.7,713.5,90.1,15.94,2.83,9.69,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 410 x 46,1",403.0,140.0,7.0,11.2,59.2,46.1,15150.0,514.0,778.7,73.4,867.8,114.4,16.27,2.95,17.46,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 410 x 53,0",403.0,177.0,7.5,10.9,68.4,53.0,18000.0,1009.0,929.7,114.0,1028.9,176.1,16.55,3.84,20.64,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 410 x 60,0",407.0,178.0,7.7,12.8,76.2,60.0,21570.0,1205.0,1066.7,135.4,1178.2,208.4,16.88,3.98,30.69,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 410 x 67,0",410.0,179.0,8.8,14.4,86.3,67.0,24678.0,1379.0,1196.0,154.1,1339.4,238.1,16.91,4.0,44.29,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 410 x 75,0",413.0,180.0,9.7,16.0,95.8,75.0,27616.0,1579.0,1337.3,173.2,1495.4,268.2,16.98,4.03,60.74,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 410 x 85,0",417.0,181.0,10.9,18.2,108.4,85.0,31600.0,1790.0,1510.0,198.0,1708.5,309.4,17.1,4.06,89.17,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 460 x 52,0",450.0,152.0,7.6,10.8,66.6,52.0,21370.0,634.0,949.8,83.5,1069.7,130.9,17.91,3.09,19.03,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 460 x 60,0",455.0,153.0,8.0,13.3,75.83,60.0,25480.0,796.0,1127.6,104.1,1265.9,162.5,18.35,3.23,31.31,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 460 x 68,0",459.0,154.0,9.1,15.4,87.6,68.0,29851.0,941.0,1300.7,122.2,1469.2,191.5,18.46,3.28,48.25,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 460 x 74,0",457.0,190.0,9.0,14.5,94.9,74.0,33415.0,1661.0,1462.4,174.8,1631.3,270.4,18.77,4.18,49.02,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 460 x 82,0",460.0,191.0,9.9,16.0,104.7,82.0,37000.0,1862.0,1615.5,195.0,1810.2,302.3,18.84,4.22,66.0,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 460 x 89,0",463.0,192.0,10.5,17.7,114.1,89.0,41105.0,2093.0,1775.6,218.0,1993.3,338.0,18.98,4.28,87.48,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 460 x 97,0",466.0,193.0,11.4,19.1,123.2,97.0,44680.0,3282.0,1920.0,236.0,2169.0,369.6,19.0,4.29,110.78,,,,,Catálogo Copromet 2024,revisar,Iy no cuadra con iy Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 460 x 106,0",469.0,194.0,12.6,20.6,134.2,106.0,48700.0,2510.0,2080.0,259.0,2368.5,404.6,19.0,4.32,141.59,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 460 x 128,0",467.0,282.0,12.2,19.6,163.0,128.0,63690.0,7333.0,2728.0,520.1,3031.1,795.3,19.73,6.7,167.45,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 460 x 177,0",482.0,286.0,16.6,26.9,226.0,177.0,91040.0,10510.0,3777.0,734.7,4262.2,1129.7,20.07,6.82,436.43,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 530 x 66,0",525.0,165.0,8.9,11.4,83.77,66.0,34971.0,857.0,1332.2,103.9,1527.2,165.1,20.46,3.2,28.1,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 530 x 72,0",524.0,207.0,9.0,10.9,91.8,72.0,40100.0,1615.0,1530.0,156.0,1725.2,243.7,20.89,4.2,30.07,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 530 x 74,0",529.0,166.0,9.7,13.6,95.28,74.0,41100.0,1041.0,1554.0,125.5,1774.2,199.2,20.76,3.31,43.1,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 530 x 82,0",528.0,209.0,9.5,13.3,104.7,82.0,47700.0,2028.0,1807.0,194.1,2027.8,301.8,21.34,4.41,47.11,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 530 x 85,0",535.0,166.0,10.3,16.5,107.7,85.0,48453.0,1263.0,1811.3,152.2,2069.1,240.7,21.21,3.42,68.0,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 530 x 92,0",533.0,209.0,10.2,15.6,117.8,92.0,55240.0,2379.0,2073.0,227.6,2329.0,353.8,21.65,4.5,70.65,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 530 x 101,0",537.0,210.0,10.9,17.4,129.0,101.0,61760.0,2690.0,2290.0,267.0,2585.9,398.6,21.8,4.57,95.43,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 530 x 109,0",539.0,211.0,11.6,18.8,139.7,109.0,67226.0,2952.0,2495.0,279.8,2792.6,435.4,21.94,4.6,119.56,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 610 x 101,0",603.0,228.0,10.5,14.9,130.3,101.0,76470.0,2951.0,2554.0,258.8,2860.4,403.1,24.31,4.76,72.4,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 610 x 113,0",608.0,228.0,11.2,17.3,145.3,113.0,87570.0,3426.0,2881.0,300.5,3250.6,467.6,24.64,4.86,105.55,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 610 x 125,0",612.0,229.0,11.9,19.6,160.1,125.0,98650.0,3933.0,3224.0,343.5,3635.0,534.2,24.89,4.96,147.13,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 610 x140,0",617.0,230.0,13.1,22.2,178.7,140.0,112000.0,4607.0,3638.8,393.8,4110.8,611.8,25.06,5.03,210.67,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 610 x 153,0",623.0,229.0,14.0,24.9,195.5,153.0,125000.0,5048.0,4019.0,434.8,4560.4,681.0,25.17,5.03,288.12,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 610 x 155,0",611.0,324.0,12.7,19.0,198.1,155.0,124000.0,10783.0,4241.7,665.6,4686.8,1020.4,25.58,7.38,187.28,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 610 x 174,0",616.0,325.0,14.0,21.6,222.8,174.0,147200.0,12374.0,4778.0,761.5,5321.0,1168.8,25.75,7.45,270.74,,,,,Catálogo Copromet 2024,sí, Zx/Zy/J aprox. desde geometría,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,100x100x3.0,100.0,100.0,3.0,,11.4,8.96,117.0,117.0,35.0,35.0,,,3.94,3.94,,,,,,Catálogo Copromet 2024,revisar,Ix no cuadra con ix; Iy no cuadra con iy,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,100x100x4.0,100.0,100.0,4.0,,14.9,11.73,226.0,226.0,45.0,45.0,,,3.89,3.89,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,100x100x5.0,100.0,100.0,5.0,,18.4,15.91,271.0,271.0,54.0,54.0,,,3.84,3.84,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,100x100x6.0,100.0,100.0,6.0,,21.6,16.98,311.0,311.0,62.0,62.0,,,3.79,3.79,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,150x150x3.0,150.0,150.0,3.0,,17.4,13.92,622.0,622.0,83.0,83.0,,,5.98,5.98,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,150x150x4.0,150.0,150.0,4.0,,22.9,18.45,807.0,807.0,108.0,108.0,,,5.93,5.93,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,150x150x5.0,150.0,150.0,5.0,,28.4,22.95,981.0,981.0,131.0,131.0,,,5.88,5.88,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,150x150x6.0,150.0,150.0,6.0,,33.6,27.38,1145.0,1145.0,153.0,153.0,,,5.84,5.84,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,200x200x3.0,200.0,200.0,3.0,,23.4,18.63,1506.0,1506.0,151.0,151.0,,,8.02,8.02,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,200x200x4.0,200.0,200.0,4.0,,30.9,24.74,1970.0,1970.0,197.0,197.0,,,7.97,7.97,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,200x200x5.0,200.0,200.0,5.0,,38.4,30.8,2410.0,2410.0,241.0,241.0,,,7.92,7.92,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,200x200x6.0,200.0,200.0,6.0,,45.6,36.81,2830.0,2830.0,283.0,283.0,,,7.88,7.88,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,250x250x4.0,250.0,250.0,4.0,,39.0,31.02,3906.0,3906.0,312.0,312.0,,,10.01,10.01,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,250x250x5.0,250.0,250.0,5.0,,48.4,38.65,4800.0,4800.0,384.0,384.0,,,9.97,9.97,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,300x300x5.0,300.0,300.0,5.0,,58.4,45.8,8414.0,8414.0,561.0,561.0,,,12.01,12.01,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,300x300x6.0,300.0,300.0,6.0,,69.4,54.7,9959.0,9959.0,664.0,664.0,,,11.96,11.96,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,150x100x3.0,150.0,100.0,3.0,,14.4,11.56,460.0,247.0,61.4,49.5,,,5.65,4.14,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,150x100x4.0,150.0,100.0,4.0,,18.9,15.31,594.0,318.0,79.2,63.6,,,5.6,4.1,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,150x100x5.0,150.0,100.0,5.0,,23.4,19.02,718.0,383.0,95.8,76.7,,,5.55,4.05,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,200x100x3.0,200.0,100.0,3.0,,17.4,13.92,924.0,318.0,92.0,63.6,,,7.29,4.27,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,200x100x4.0,200.0,100.0,4.0,,22.9,18.45,1200.0,410.0,120.0,82.1,,,7.23,4.23,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,200x100x5.0,200.0,100.0,5.0,,28.4,22.95,1460.0,496.0,146.0,99.2,,,7.17,4.18,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,200x150x3.0,200.0,150.0,3.0,,20.4,16.24,1215.0,785.0,122.0,104.6,,,7.72,6.2,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,200x150x4.0,200.0,150.0,4.0,,26.9,21.6,1580.0,1020.0,158.0,136.0,,,7.66,6.15,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,200x150x5.0,200.0,150.0,5.0,,33.4,26.87,1930.0,1240.0,193.0,166.0,,,7.61,6.11,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,200x150x6.0,200.0,150.0,6.0,,39.6,32.1,2270.0,1460.0,227.0,194.0,,,7.56,6.06,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,250x100x3.0,250.0,100.0,3.0,,20.64,16.27,1642.0,394.0,131.2,78.9,,,8.92,4.37,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,250x100x4.0,250.0,100.0,4.0,,27.4,21.6,2155.0,513.0,172.4,102.6,,,8.88,4.33,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,250x100x5.0,250.0,100.0,5.0,,34.0,26.69,2653.0,625.0,212.2,125.1,,,8.83,4.29,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,250x200x5.0,250.0,200.0,5.0,,44.0,34.72,4154.0,2949.0,332.3,332.3,,,9.72,8.19,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,300x100x3.0,300.0,100.0,3.0,,23.6,18.63,2594.0,465.0,172.9,93.0,,,10.47,4.44,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,300x100x4.0,300.0,100.0,4.0,,31.36,24.74,3412.0,605.0,227.5,121.0,,,10.43,4.39,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,300x100x5.0,300.0,100.0,5.0,,39.0,30.8,4208.0,738.3,280.6,147.7,,,10.39,4.35,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,300x150x5.0,300.0,150.0,5.0,,43.4,38.52,5151.0,1770.0,343.0,235.9,,,10.9,6.39,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,300x150x6.0,300.0,150.0,6.0,,51.6,41.52,6070.0,2077.0,405.0,277.0,,,10.84,6.34,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,300x200x5.0,300.0,200.0,5.0,,48.4,38.65,6238.0,3359.0,416.0,335.9,,,11.36,8.33,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,300x200x6.0,300.0,200.0,6.0,,57.6,46.23,7366.0,3959.0,491.0,395.9,,,11.31,8.29,,,,,,Catálogo Copromet 2024,sí,,
Copromet Tubo cuadrado/rectangular ASTM A500 (Copromet),HSS rectangular,300x250x5.0,300.0,250.0,5.0,,54.0,42.58,7472.0,5654.0,498.1,452.4,,,11.76,10.23,,,,,,Catálogo Copromet 2024,sí,,
Copromet W (ASTM A6),I/H ala ancha,"W 310 x 86,0",310.0,254.0,9.1,16.3,110.0,86.0,19850.0,4455.0,128.0,35.08,1391.0,531.5,13.43,6.36,80.3,,,,,Catálogo Copromet 2024,revisar,Tabla con typos en Wx/Wy; valores recalculados desde Ix/(h/2) e Iy/(b/2) Zx/Zy/J aprox. desde geometría,
Copromet W (ASTM A6),I/H ala ancha,"W 310 x 32,9",349.0,127.0,5.8,8.5,41.7,32.9,8258.0,291.0,47.3,4.58,527.4,71.3,14.09,2.63,7.36,,,,,Catálogo Copromet 2024,revisar,"Tabla con typos en Ix/Wx/Wy; valores recalculados desde ix, A, h, b e Iy/(b/2) Zx/Zy/J aprox. desde geometría",
Cintac Tubo redondo ASTM A500,HSS circular,Ø12.7x0.9,12.7,12.7,0.9,0.9,0.33,0.26,0.06,0.06,0.09,0.09,,,0.42,0.42,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø12.7x1.2,12.7,12.7,1.2,1.2,0.43,0.34,0.07,0.07,0.11,0.11,,,0.41,0.41,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø12.7x1.5,12.7,12.7,1.5,1.5,0.53,0.41,0.08,0.08,0.13,0.13,,,0.4,0.4,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø15.88x0.9,15.88,15.88,0.9,0.9,0.42,0.33,0.12,0.12,0.15,0.15,,,0.53,0.53,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø15.88x1,15.88,15.88,1.0,1.0,0.47,0.37,0.13,0.13,0.16,0.16,,,0.53,0.53,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø15.88x1.2,15.88,15.88,1.2,1.2,0.55,0.43,0.15,0.15,0.19,0.19,,,0.52,0.52,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø15.88x1.5,15.88,15.88,1.5,1.5,0.68,0.53,0.18,0.18,0.22,0.22,,,0.51,0.51,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø15.88x2,15.88,15.88,2.0,2.0,0.87,0.68,0.21,0.21,0.27,0.27,,,0.5,0.5,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø19.05x0.9,19.05,19.05,0.9,0.9,0.51,0.4,0.21,0.21,0.22,0.22,,,0.64,0.64,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø19.05x1,19.05,19.05,1.0,1.0,0.57,0.45,0.23,0.23,0.24,0.24,,,0.64,0.64,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø19.05x1.2,19.05,19.05,1.2,1.2,0.67,0.53,0.27,0.27,0.28,0.28,,,0.63,0.63,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø19.05x1.5,19.05,19.05,1.5,1.5,0.83,0.65,0.32,0.32,0.34,0.34,,,0.62,0.62,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø19.05x2,19.05,19.05,2.0,2.0,1.07,0.84,0.39,0.39,0.41,0.41,,,0.61,0.61,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø22.22x0.9,22.22,22.22,0.9,0.9,0.6,0.47,0.34,0.34,0.31,0.31,,,0.75,0.75,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø22.22x1,22.22,22.22,1.0,1.0,0.67,0.52,0.38,0.38,0.34,0.34,,,0.75,0.75,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø22.22x1.2,22.22,22.22,1.2,1.2,0.79,0.62,0.44,0.44,0.4,0.4,,,0.74,0.74,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø22.22x1.5,22.22,22.22,1.5,1.5,0.98,0.77,0.53,0.53,0.47,0.47,,,0.73,0.73,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø38.1x1,38.1,38.1,1.0,1.0,1.17,0.91,2.01,2.01,1.05,1.05,,,1.31,1.31,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø38.1x1.2,38.1,38.1,1.2,1.2,1.39,1.09,2.37,2.37,1.24,1.24,,,1.31,1.31,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø38.1x1.5,38.1,38.1,1.5,1.5,1.72,1.35,2.89,2.89,1.52,1.52,,,1.3,1.3,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø38.1x2,38.1,38.1,2.0,2.0,2.27,1.78,3.71,3.71,1.95,1.95,,,1.28,1.28,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø44.45x1,44.45,44.45,1.0,1.0,1.37,1.07,3.22,3.22,1.45,1.45,,,1.54,1.54,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø44.45x1.2,44.45,44.45,1.2,1.2,1.63,1.28,3.82,3.82,1.72,1.72,,,1.53,1.53,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø44.45x1.5,44.45,44.45,1.5,1.5,2.02,1.59,4.67,4.67,2.1,2.1,,,1.52,1.52,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø44.45x2,44.45,44.45,2.0,2.0,2.67,2.09,6.02,6.02,2.71,2.71,,,1.5,1.5,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø50.8x1,50.8,50.8,1.0,1.0,1.56,1.23,4.85,4.85,1.91,1.91,,,1.76,1.76,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø50.8x1.2,50.8,50.8,1.2,1.2,1.87,1.47,5.75,5.75,2.27,2.27,,,1.75,1.75,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø50.8x1.5,50.8,50.8,1.5,1.5,2.32,1.82,7.06,7.06,2.78,2.78,,,1.74,1.74,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø50.8x2,50.8,50.8,2.0,2.0,3.07,2.41,9.14,9.14,3.6,3.6,,,1.73,1.73,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø50.8x3,50.8,50.8,3.0,3.0,4.51,3.54,12.92,12.92,5.09,5.09,,,1.69,1.69,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø63.5x1.5,63.5,63.5,1.5,1.5,2.92,2.29,14.05,14.05,4.42,4.42,,,2.19,2.19,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø63.5x2,63.5,63.5,2.0,2.0,3.86,3.03,18.29,18.29,5.76,5.76,,,2.18,2.18,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø63.5x3,63.5,63.5,3.0,3.0,5.7,4.48,26.15,26.15,8.24,8.24,,,2.14,2.14,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø76.2x1.5,76.2,76.2,1.5,1.5,3.52,2.76,24.56,24.56,6.45,6.45,,,2.64,2.64,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø76.2x2,76.2,76.2,2.0,2.0,4.66,3.66,32.11,32.11,8.43,8.43,,,2.62,2.62,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø76.2x3,76.2,76.2,3.0,3.0,6.9,5.42,46.29,46.29,12.15,12.15,,,2.59,2.59,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø76.2x4,76.2,76.2,4.0,4.0,9.07,7.12,59.3,59.3,15.56,15.56,,,2.56,2.56,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø88.9x2,88.9,88.9,2.0,2.0,5.46,4.29,51.57,51.57,11.6,11.6,,,3.07,3.07,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø88.9x3,88.9,88.9,3.0,3.0,8.1,6.36,74.76,74.76,16.82,16.82,,,3.04,3.04,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø101.6x2,101.6,101.6,2.0,2.0,6.26,4.91,77.63,77.63,15.28,15.28,,,3.52,3.52,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø101.6x3,101.6,101.6,3.0,3.0,9.29,7.29,113.04,113.04,22.25,22.25,,,3.49,3.49,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø101.6x4,101.6,101.6,4.0,4.0,12.26,9.63,146.28,146.28,28.8,28.8,,,3.45,3.45,,,,,,Catálogo técnico Cintac,sí,,
Cintac Tubo redondo ASTM A500,HSS circular,Ø101.6x5,101.6,101.6,5.0,5.0,15.17,11.9,177.47,177.47,34.93,34.93,,,3.42,3.42,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□12x1,12.0,12.0,1.0,1.0,0.41,0.33,0.08,0.08,0.13,0.13,,,0.44,0.44,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□15x1,15.0,15.0,1.0,1.0,0.53,0.42,0.17,0.17,0.23,0.23,,,0.56,0.56,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□15x1.5,15.0,15.0,1.5,1.5,0.75,0.59,0.22,0.22,0.29,0.29,,,0.54,0.54,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□20x1,20.0,20.0,1.0,1.0,0.73,0.58,0.43,0.43,0.43,0.43,,,0.77,0.77,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□20x1.5,20.0,20.0,1.5,1.5,1.05,0.83,0.58,0.58,0.58,0.58,,,0.74,0.74,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□20x2,20.0,20.0,2.0,2.0,1.34,1.05,0.69,0.69,0.69,0.69,,,0.72,0.72,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□25x1,25.0,25.0,1.0,1.0,0.93,0.73,0.88,0.88,0.71,0.71,,,0.97,0.97,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□25x1.5,25.0,25.0,1.5,1.5,1.35,1.06,1.21,1.21,0.97,0.97,,,0.95,0.95,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□25x2,25.0,25.0,2.0,2.0,1.74,1.36,1.48,1.48,1.18,1.18,,,0.92,0.92,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□30x1,30.0,30.0,1.0,1.0,1.13,0.89,1.57,1.57,1.05,1.05,,,1.18,1.18,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□30x1.5,30.0,30.0,1.5,1.5,1.65,1.3,2.19,2.19,1.46,1.46,,,1.15,1.15,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□30x2,30.0,30.0,2.0,2.0,2.14,1.68,2.71,2.71,1.81,1.81,,,1.13,1.13,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□40x1,40.0,40.0,1.0,1.0,1.53,1.2,3.85,3.85,1.93,1.93,,,1.59,1.59,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□40x1.5,40.0,40.0,1.5,1.5,2.25,1.77,5.48,5.48,2.74,2.74,,,1.56,1.56,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□40x2,40.0,40.0,2.0,2.0,2.94,2.31,6.93,6.93,3.46,3.46,,,1.54,1.54,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□40x3,40.0,40.0,3.0,3.0,4.21,3.3,9.28,9.28,4.64,4.64,,,1.48,1.48,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□50x1.5,50.0,50.0,1.5,1.5,2.85,2.24,11.06,11.06,4.42,4.42,,,1.97,1.97,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□50x2,50.0,50.0,2.0,2.0,3.74,2.93,14.13,14.13,5.65,5.65,,,1.94,1.94,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□50x3,50.0,50.0,3.0,3.0,5.41,4.25,19.41,19.41,7.76,7.76,,,1.89,1.89,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□50x5,50.0,50.0,5.0,5.0,8.36,6.56,26.78,26.78,10.71,10.71,,,1.79,1.79,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□50x6,50.0,50.0,6.0,6.0,9.63,7.56,29.03,29.03,11.61,11.61,,,1.74,1.74,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□75x2,75.0,75.0,2.0,2.0,5.74,4.5,50.47,50.47,13.46,13.46,,,2.97,2.97,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□75x3,75.0,75.0,3.0,3.0,8.41,6.6,71.54,71.54,19.08,19.08,,,2.92,2.92,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□75x4,75.0,75.0,4.0,4.0,10.95,8.59,89.98,89.98,24.0,24.0,,,2.87,2.87,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□75x5,75.0,75.0,5.0,5.0,13.36,10.48,105.92,105.92,28.25,28.25,,,2.82,2.82,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□75x6,75.0,75.0,6.0,6.0,15.63,12.27,119.48,119.48,31.86,31.86,,,2.76,2.76,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□100x2,100.0,100.0,2.0,2.0,7.74,6.07,122.99,122.99,24.6,24.6,,,3.99,3.99,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□100x3,100.0,100.0,3.0,3.0,11.41,8.96,176.95,176.95,35.39,35.39,,,3.94,3.94,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□100x4,100.0,100.0,4.0,4.0,14.95,11.73,226.09,226.09,45.22,45.22,,,3.89,3.89,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□100x5,100.0,100.0,5.0,5.0,18.36,14.41,270.57,270.57,54.11,54.11,,,3.84,3.84,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□100x6,100.0,100.0,6.0,6.0,21.63,16.98,310.55,310.55,62.11,62.11,,,3.79,3.79,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□135x4,135.0,135.0,4.0,4.0,20.55,16.13,581.38,581.38,86.13,86.13,,,5.32,5.32,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□135x5,135.0,135.0,5.0,5.0,25.36,19.9,704.23,704.23,104.33,104.33,,,5.27,5.27,,,,,,Catálogo técnico Cintac,sí,,
Cintac Cuadrado ASTM A500,HSS cuadrado,□135x6,135.0,135.0,6.0,6.0,30.03,23.58,818.5,818.5,121.26,121.26,,,5.22,5.22,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,20x10x1,20.0,10.0,1.0,1.0,0.53,0.42,0.25,0.08,0.25,0.17,,,0.69,0.39,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,20x10x1.5,20.0,10.0,1.5,1.5,0.75,0.59,0.32,0.1,0.32,0.21,,,0.66,0.37,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,25x15x1,25.0,15.0,1.0,1.0,0.73,0.58,0.59,0.27,0.48,0.36,,,0.9,0.6,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,25x15x1.5,25.0,15.0,1.5,1.5,1.05,0.83,0.8,0.35,0.64,0.47,,,0.87,0.58,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,25x15x2,25.0,15.0,2.0,2.0,1.34,1.05,0.95,0.41,0.76,0.55,,,0.84,0.56,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,30x20x1,30.0,20.0,1.0,1.0,0.93,0.73,1.15,0.61,0.77,0.61,,,1.11,0.81,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,30x20x1.5,30.0,20.0,1.5,1.5,1.35,1.06,1.58,0.84,1.05,0.84,,,1.08,0.79,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,30x20x2,30.0,20.0,2.0,2.0,1.74,1.36,1.93,1.01,1.28,1.01,,,1.05,0.76,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,40x20x1,40.0,20.0,1.0,1.0,1.13,0.89,2.33,0.79,1.17,0.79,,,1.43,0.84,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,40x20x1.5,40.0,20.0,1.5,1.5,1.65,1.3,3.26,1.09,1.63,1.09,,,1.4,0.81,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,40x20x2,40.0,20.0,2.0,2.0,2.14,1.68,4.04,1.33,2.02,1.33,,,1.37,0.79,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,40x30x1,40.0,30.0,1.0,1.0,1.33,1.05,3.09,1.99,1.55,1.33,,,1.52,1.22,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,40x30x1.5,40.0,30.0,1.5,1.5,1.95,1.53,4.37,2.8,2.19,1.87,,,1.5,1.2,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,40x30x2,40.0,30.0,2.0,2.0,2.54,1.99,5.48,3.5,2.74,2.33,,,1.47,1.17,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,50x20x1,50.0,20.0,1.0,1.0,1.33,1.05,4.08,0.97,1.63,0.97,,,1.75,0.85,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,50x20x1.5,50.0,20.0,1.5,1.5,1.95,1.53,5.76,1.35,2.31,1.35,,,1.72,0.83,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,50x20x2,50.0,20.0,2.0,2.0,2.54,1.99,7.22,1.66,2.89,1.66,,,1.69,0.81,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,50x30x1,50.0,30.0,1.0,1.0,1.53,1.2,5.28,2.41,2.11,1.61,,,1.86,1.25,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,50x30x1.5,50.0,30.0,1.5,1.5,2.25,1.77,7.53,3.41,3.01,2.27,,,1.83,1.23,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,50x30x2,50.0,30.0,2.0,2.0,2.94,2.31,9.52,4.28,3.81,2.85,,,1.8,1.21,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,50x30x3,50.0,30.0,3.0,3.0,4.21,3.3,12.78,5.66,5.11,3.77,,,1.74,1.16,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,60x40x2,60.0,40.0,2.0,2.0,3.74,2.93,18.39,9.81,6.13,4.91,,,2.22,1.62,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,60x40x3,60.0,40.0,3.0,3.0,5.41,4.25,25.31,13.38,8.44,6.69,,,2.16,1.57,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,60x40x4,60.0,40.0,4.0,4.0,6.95,5.45,30.83,16.15,10.28,8.08,,,2.11,1.52,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,70x30x2,70.0,30.0,2.0,2.0,3.74,2.93,22.2,5.85,6.34,3.9,,,2.44,1.25,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,70x30x3,70.0,30.0,3.0,3.0,5.41,4.25,30.5,7.84,8.71,5.23,,,2.37,1.2,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,70x30x4,70.0,30.0,4.0,4.0,6.95,5.45,37.07,9.3,10.59,6.2,,,2.31,1.16,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,80x40x2,80.0,40.0,2.0,2.0,4.54,3.56,37.33,12.7,9.33,6.35,,,2.87,1.67,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,80x40x3,80.0,40.0,3.0,3.0,6.61,5.19,52.16,17.49,13.04,8.74,,,2.81,1.63,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,80x40x4,80.0,40.0,4.0,4.0,8.55,6.71,64.59,21.33,16.15,10.67,,,2.75,1.58,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,100x50x2,100.0,50.0,2.0,2.0,5.74,4.5,74.94,25.65,14.99,10.26,,,3.61,2.11,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,100x50x3,100.0,50.0,3.0,3.0,8.41,6.6,106.34,35.97,21.27,14.39,,,3.56,2.07,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,100x50x4,100.0,50.0,4.0,4.0,10.59,8.59,133.88,44.76,26.78,17.9,,,3.5,2.02,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,100x50x5,100.0,50.0,5.0,5.0,13.36,10.48,157.7,52.09,31.54,20.83,,,3.44,1.97,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,150x50x2,150.0,50.0,2.0,2.0,7.74,6.07,207.45,37.17,27.66,14.87,,,5.18,2.19,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,150x50x3,150.0,50.0,3.0,3.0,11.41,8.96,298.35,52.54,39.78,21.02,,,5.11,2.15,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,150x50x4,150.0,50.0,4.0,4.0,14.95,11.73,380.98,65.92,50.8,26.37,,,5.05,2.1,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,150x50x5,150.0,50.0,5.0,5.0,18.36,14.41,455.54,77.4,60.74,30.96,,,4.98,2.05,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,200x70x4,200.0,70.0,4.0,4.0,20.55,16.13,968.54,185.17,96.85,52.91,,,6.87,3.0,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,200x70x5,200.0,70.0,5.0,5.0,25.36,19.9,1172.89117,221.55,29.0,63.3,,,6.8,2.96,,,,,,Catálogo técnico Cintac,sí,,
Cintac Rectangular ASTM A500,HSS rectangular,200x70x6,200.0,70.0,6.0,6.0,30.03,23.58,1362.69136,254.26,27.0,72.65,,,6.74,2.91,,,,,,Catálogo técnico Cintac,sí,,
Cintac Costanera Cintac 2.7,C labio (costanera),C 80x40x15x2,80.0,40.0,2.0,2.0,3.54,2.78,35.3,8.07,8.81,3.18,,,3.16,1.51,,,,,,Catálogo técnico Cintac,sí,,
Cintac Costanera Cintac 2.7,C labio (costanera),C 80x40x15x3,80.0,40.0,3.0,3.0,5.11,4.01,49.0,10.85,12.26,4.27,,,3.1,1.46,,,,,,Catálogo técnico Cintac,sí,,
Cintac Costanera Cintac 2.7,C labio (costanera),C 80x40x15x4,80.0,40.0,4.0,4.0,6.55,5.14,60.4,12.88,15.11,5.05,,,3.04,1.4,,,,,,Catálogo técnico Cintac,sí,,
Cintac Costanera Cintac 2.7,C labio (costanera),C 100x50x15x2,100.0,50.0,2.0,2.0,4.34,3.4,69.2,14.98,13.85,4.57,,,4.0,1.86,,,,,,Catálogo técnico Cintac,sí,,
Cintac Costanera Cintac 2.7,C labio (costanera),C 100x50x15x3,100.0,50.0,3.0,3.0,6.31,4.95,97.8,20.52,19.56,6.25,,,3.94,1.8,,,,,,Catálogo técnico Cintac,sí,,
Cintac Costanera Cintac 2.7,C labio (costanera),C 100x50x15x4,100.0,50.0,4.0,4.0,8.15,6.4,122.5,24.86,24.5,7.55,,,3.88,1.75,,,,,,Catálogo técnico Cintac,sí,,
Cintac Costanera Cintac 2.7,C labio (costanera),C 125x50x15x2,125.0,50.0,2.0,2.0,4.84,3.8,116.4,16.16,18.63,4.7,,,4.91,1.83,,,,,,Catálogo técnico Cintac,sí,,
Cintac Costanera Cintac 2.7,C labio (costanera),C 125x50x15x3,125.0,50.0,3.0,3.0,7.06,5.54,165.5,22.16,26.48,6.43,,,4.84,1.77,,,,,,Catálogo técnico Cintac,sí,,
Cintac Costanera Cintac 2.7,C labio (costanera),C 125x50x15x4,125.0,50.0,4.0,4.0,9.15,7.18,208.7,26.89,33.39,7.78,,,4.78,1.71,,,,,,Catálogo técnico Cintac,sí,,
Cintac Costanera Cintac 2.7,C labio (costanera),C 150x50x15x2,150.0,50.0,2.0,2.0,5.34,4.19,178.7,17.13,23.83,4.79,,,5.79,1.79,,,,,,Catálogo técnico Cintac,sí,,
Cintac Costanera Cintac 2.7,C labio (costanera),C 150x50x15x3,150.0,50.0,3.0,3.0,7.81,6.13,255.3,23.49,34.03,6.56,,,5.72,1.73,,,,,,Catálogo técnico Cintac,sí,,
Cintac Costanera Cintac 2.7,C labio (costanera),C 150x50x15x4,150.0,50.0,4.0,4.0,10.15,7.97,323.5,28.51,43.13,7.95,,,5.65,1.68,,,,,,Catálogo técnico Cintac,sí,,
Cintac Costanera Cintac 2.7,C labio (costanera),C 150x75x15x2,150.0,75.0,2.0,2.0,6.34,4.97,233.5,46.34,31.13,9.03,,,6.07,2.7,,,,,,Catálogo técnico Cintac,sí,,
Cintac Costanera Cintac 2.7,C labio (costanera),C 150x75x15x3,150.0,75.0,3.0,3.0,9.31,7.31,336.3,65.14,44.84,12.67,,,6.01,2.65,,,,,,Catálogo técnico Cintac,sí,,
Cintac Costanera Cintac 2.7,C labio (costanera),C 150x75x15x4,150.0,75.0,4.0,4.0,12.15,9.54,430.1,81.19,57.34,15.76,,,5.95,2.59,,,,,,Catálogo técnico Cintac,sí,,
Cintac Costanera Cintac 2.7,C labio (costanera),C 175x75x15x3,175.0,75.0,3.0,3.0,10.06,7.9,480.4,68.52,54.9,12.91,,,6.91,2.61,,,,,,Catálogo técnico Cintac,sí,,
Cintac Costanera Cintac 2.7,C labio (costanera),C 200x50x15x2,200.0,50.0,2.0,2.0,6.34,4.97,354.9,18.6,35.49,4.91,,,7.48,1.71,,,,,,Catálogo técnico Cintac,sí,,
Cintac Costanera Cintac 2.7,C labio (costanera),C 200x50x15x3,200.0,50.0,3.0,3.0,9.31,7.31,510.4,25.51,51.04,6.74,,,7.4,1.66,,,,,,Catálogo técnico Cintac,sí,,
Cintac Costanera Cintac 2.7,C labio (costanera),C 200x50x15x4,200.0,50.0,4.0,4.0,12.15,9.54,651.4,30.97,65.14,8.18,,,7.32,1.6,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),50x25x2,50.0,25.0,2.0,2.0,1.87,1.47,7.07,1.13,2.83,0.63,,,1.94,0.78,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),50x25x3,50.0,25.0,3.0,3.0,2.7,2.12,9.7,1.57,3.88,0.91,,,1.89,0.76,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),80x40x2,80.0,40.0,2.0,2.0,3.07,2.41,30.84,4.89,7.71,1.68,,,3.17,1.26,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),80x40x3,80.0,40.0,3.0,3.0,4.5,3.54,43.88,7.01,10.97,2.45,,,3.12,1.25,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),80x40x4,80.0,40.0,4.0,4.0,5.87,4.61,55.41,8.92,13.85,3.17,,,3.07,1.23,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),80x40x5,80.0,40.0,5.0,5.0,7.18,5.63,65.51,10.62,16.38,3.84,,,3.02,1.22,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),100x50x2,100.0,50.0,2.0,2.0,3.87,3.04,61.5,9.72,12.3,2.66,,,3.99,1.58,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),100x50x3,100.0,50.0,3.0,3.0,5.7,4.48,88.47,14.07,17.69,3.89,,,3.94,1.57,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),100x50x4,100.0,50.0,4.0,4.0,7.47,5.87,113.04,18.08,22.61,5.07,,,3.89,1.56,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),100x50x5,100.0,50.0,5.0,5.0,9.18,7.2,135.28,21.77,27.06,6.19,,,3.84,1.54,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),100x50x6,100.0,50.0,6.0,6.0,10.82,8.49,155.27,25.14,31.05,7.25,,,3.79,1.52,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),100x75x3,100.0,75.0,3.0,3.0,7.2,5.66,123.76,42.91,24.75,8.42,,,4.14,2.44,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),100x75x4,100.0,75.0,4.0,4.0,9.47,7.44,159.12,55.68,31.82,11.03,,,4.1,2.42,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),100x75x5,100.0,75.0,5.0,5.0,11.68,9.17,191.69,67.7,38.34,13.55,,,4.05,2.41,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),125x50x2,125.0,50.0,2.0,2.0,4.37,3.43,103.19,10.4,16.51,2.74,,,4.86,1.54,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),125x50x3,125.0,50.0,3.0,3.0,6.45,5.07,149.29,15.08,23.89,4.02,,,4.81,1.53,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),125x50x4,125.0,50.0,4.0,4.0,8.47,6.65,191.86,19.43,30.7,5.24,,,4.76,1.51,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),125x50x5,125.0,50.0,5.0,5.0,10.43,8.19,230.99,23.44,36.96,6.4,,,4.71,1.5,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),150x50x3,150.0,50.0,3.0,3.0,7.2,5.66,230.29,15.89,30.7,4.11,,,5.65,1.49,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),150x50x4,150.0,50.0,4.0,4.0,9.47,7.44,297.17,20.49,39.62,5.36,,,5.6,1.47,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),150x50x5,150.0,50.0,5.0,5.0,11.68,9.17,359.29,24.76,47.91,6.55,,,5.55,1.46,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),150x50x6,150.0,50.0,6.0,6.0,13.82,10.85,416.76,28.7,55.57,7.68,,,5.49,1.44,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),175x50x3,175.0,50.0,3.0,3.0,7.95,6.24,333.81,16.54,38.15,4.18,,,6.48,1.44,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),175x50x4,175.0,50.0,4.0,4.0,10.47,8.22,432.09,21.35,49.38,5.45,,,6.42,1.43,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),175x50x5,175.0,50.0,5.0,5.0,12.93,10.15,524.1,25.82,59.9,6.66,,,6.37,1.41,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),200x50x2,200.0,50.0,2.0,2.0,5.87,4.61,316.47,11.75,31.65,2.88,,,7.34,1.42,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),200x50x3,200.0,50.0,3.0,3.0,8.7,6.83,462.19,17.08,46.22,4.23,,,7.29,1.4,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),200x50x4,200.0,50.0,4.0,4.0,11.47,9.01,599.77,22.06,59.98,5.52,,,7.23,1.39,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),200x50x5,200.0,50.0,5.0,5.0,14.18,11.13,729.33,26.69,72.93,6.75,,,7.17,1.37,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),200x50x6,200.0,50.0,6.0,6.0,16.82,13.2,851.04,30.99,85.1,7.93,,,7.11,1.36,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),200x75x3,200.0,75.0,3.0,3.0,10.2,8.01,607.73,53.64,60.77,9.31,,,7.72,2.29,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),200x75x4,200.0,75.0,4.0,4.0,13.47,10.58,791.85,69.95,79.18,12.24,,,7.67,2.28,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),200x75x5,200.0,75.0,5.0,5.0,16.68,13.09,966.99,85.49,96.7,15.07,,,7.61,2.26,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),200x75x6,200.0,75.0,6.0,6.0,19.82,15.56,1133.31,100.27,113.33,17.82,,,7.56,2.25,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),50x2x3,50.0,2.0,3.0,3.0,10.2,8.01,802.96,17.92,64.24,4.31,,,8.87,1.33,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),50x2x4,50.0,2.0,4.0,4.0,13.47,10.58,1045.86,23.16,83.67,5.63,,,8.81,1.31,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),50x2x5,50.0,2.0,5.0,5.0,16.68,13.09,1276.69,28.05,102.14,6.89,,,8.75,1.3,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),50x2x6,50.0,2.0,6.0,6.0,19.82,15.56,1495.62,32.59,119.65,8.09,,,8.69,1.28,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),250x75x3,250.0,75.0,3.0,3.0,11.7,9.19,1031.74,56.94,82.54,9.55,,,9.39,2.21,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),250x75x4,250.0,75.0,4.0,4.0,15.47,12.15,1348.44,74.31,107.88,12.55,,,9.34,2.19,,,,,,Catálogo técnico Cintac,sí,,
Cintac Canal Cintac 2.7,C (canal),250x75x5,250.0,75.0,5.0,5.0,19.18,15.05,1651.85,90.9,132.15,15.47,,,9.28,2.18,,,,,,Catálogo técnico Cintac,sí,,
Cintac Ángulo Cintac 2.7,L (ángulo),20x20x2,20.0,20.0,2.0,2.0,0.73,0.58,0.28,0.28,0.2,0.2,,,0.62,0.62,,,,,,Catálogo técnico Cintac,sí,,0.37
Cintac Ángulo Cintac 2.7,L (ángulo),20x20x3,20.0,20.0,3.0,3.0,1.05,0.83,0.38,0.38,0.28,0.28,,,0.6,0.6,,,,,,Catálogo técnico Cintac,sí,,0.35
Cintac Ángulo Cintac 2.7,L (ángulo),25x25x2,25.0,25.0,2.0,2.0,0.93,0.73,0.56,0.56,0.32,0.32,,,0.78,0.78,,,,,,Catálogo técnico Cintac,sí,,0.47
Cintac Ángulo Cintac 2.7,L (ángulo),25x25x3,25.0,25.0,3.0,3.0,1.35,1.06,0.79,0.79,0.45,0.45,,,0.76,0.76,,,,,,Catálogo técnico Cintac,sí,,0.45
Cintac Ángulo Cintac 2.7,L (ángulo),30x30x2,30.0,30.0,2.0,2.0,1.13,0.89,1.0,1.0,0.46,0.46,,,0.94,0.94,,,,,,Catálogo técnico Cintac,sí,,0.57
Cintac Ángulo Cintac 2.7,L (ángulo),30x30x3,30.0,30.0,3.0,3.0,1.65,1.3,1.41,1.41,0.67,0.67,,,0.92,0.92,,,,,,Catálogo técnico Cintac,sí,,0.55
Cintac Ángulo Cintac 2.7,L (ángulo),30x30x4,30.0,30.0,4.0,4.0,2.14,1.68,1.76,1.76,0.86,0.86,,,0.91,0.91,,,,,,Catálogo técnico Cintac,sí,,0.53
Cintac Ángulo Cintac 2.7,L (ángulo),40x40x2,40.0,40.0,2.0,2.0,1.53,1.2,2.44,2.44,0.84,0.84,,,1.26,1.26,,,,,,Catálogo técnico Cintac,sí,,0.78
Cintac Ángulo Cintac 2.7,L (ángulo),40x40x3,40.0,40.0,3.0,3.0,2.25,1.77,3.5,3.5,1.22,1.22,,,1.25,1.25,,,,,,Catálogo técnico Cintac,sí,,0.76
Cintac Ángulo Cintac 2.7,L (ángulo),40x40x4,40.0,40.0,4.0,4.0,2.94,2.31,4.46,4.46,1.58,1.58,,,1.23,1.23,,,,,,Catálogo técnico Cintac,sí,,0.74
Cintac Ángulo Cintac 2.7,L (ángulo),40x40x5,40.0,40.0,5.0,5.0,3.59,2.82,5.31,5.31,1.92,1.92,,,1.22,1.22,,,,,,Catálogo técnico Cintac,sí,,0.71
Cintac Ángulo Cintac 2.7,L (ángulo),40x40x6,40.0,40.0,6.0,6.0,4.21,3.3,6.06,6.06,2.23,2.23,,,1.2,1.2,,,,,,Catálogo técnico Cintac,sí,,0.69
Cintac Ángulo Cintac 2.7,L (ángulo),50x50x2,50.0,50.0,2.0,2.0,1.93,1.52,4.86,4.86,1.33,1.33,,,1.58,1.58,,,,,,Catálogo técnico Cintac,sí,,0.98
Cintac Ángulo Cintac 2.7,L (ángulo),50x50x3,50.0,50.0,3.0,3.0,2.85,2.24,7.03,7.03,1.95,1.95,,,1.57,1.57,,,,,,Catálogo técnico Cintac,sí,,0.96
Cintac Ángulo Cintac 2.7,L (ángulo),50x50x4,50.0,50.0,4.0,4.0,3.74,2.93,9.04,9.04,2.54,2.54,,,1.56,1.56,,,,,,Catálogo técnico Cintac,sí,,0.94
Cintac Ángulo Cintac 2.7,L (ángulo),50x50x5,50.0,50.0,5.0,5.0,4.59,3.6,10.88,10.88,3.1,3.1,,,1.54,1.54,,,,,,Catálogo técnico Cintac,sí,,0.92
Cintac Ángulo Cintac 2.7,L (ángulo),50x50x6,50.0,50.0,6.0,6.0,5.41,4.25,12.57,12.57,3.62,3.62,,,1.52,1.52,,,,,,Catálogo técnico Cintac,sí,,0.9
Cintac Ángulo Cintac 2.7,L (ángulo),65x65x3,65.0,65.0,3.0,3.0,3.75,2.95,15.84,15.84,3.34,3.34,,,2.05,2.05,,,,,,Catálogo técnico Cintac,sí,,1.27
Cintac Ángulo Cintac 2.7,L (ángulo),65x65x4,65.0,65.0,4.0,4.0,4.94,3.88,20.54,20.54,4.38,4.38,,,2.04,2.04,,,,,,Catálogo técnico Cintac,sí,,1.25
Cintac Ángulo Cintac 2.7,L (ángulo),65x65x5,65.0,65.0,5.0,5.0,6.09,4.78,24.97,24.97,5.38,5.38,,,2.02,2.02,,,,,,Catálogo técnico Cintac,sí,,1.23
Cintac Ángulo Cintac 2.7,L (ángulo),65x65x6,65.0,65.0,6.0,6.0,7.21,5.66,29.11,29.11,6.34,6.34,,,2.01,2.01,,,,,,Catálogo técnico Cintac,sí,,1.21
Cintac Ángulo Cintac 2.7,L (ángulo),80x80x3,80.0,80.0,3.0,3.0,4.65,3.65,29.99,29.99,5.12,5.12,,,2.54,2.54,,,,,,Catálogo técnico Cintac,sí,,1.57
Cintac Ángulo Cintac 2.7,L (ángulo),80x80x4,80.0,80.0,4.0,4.0,6.14,4.82,39.1,39.1,6.72,6.72,,,2.52,2.52,,,,,,Catálogo técnico Cintac,sí,,1.56
Cintac Ángulo Cintac 2.7,L (ángulo),80x80x5,80.0,80.0,5.0,5.0,7.59,5.96,47.79,47.79,8.28,8.28,,,2.51,2.51,,,,,,Catálogo técnico Cintac,sí,,1.54
Cintac Ángulo Cintac 2.7,L (ángulo),80x80x6,80.0,80.0,6.0,6.0,9.01,7.07,56.05,56.05,9.8,9.8,,,2.49,2.49,,,,,,Catálogo técnico Cintac,sí,,1.52
Cintac Ángulo Cintac 2.7,L (ángulo),100x100x3,100.0,100.0,3.0,3.0,5.85,4.59,59.35,59.35,8.06,8.06,,,3.18,3.18,,,,,,Catálogo técnico Cintac,sí,,1.98
Cintac Ángulo Cintac 2.7,L (ángulo),100x100x4,100.0,100.0,4.0,4.0,7.74,6.07,77.75,77.75,10.63,10.63,,,3.17,3.17,,,,,,Catálogo técnico Cintac,sí,,1.96
Cintac Ángulo Cintac 2.7,L (ángulo),100x100x5,100.0,100.0,5.0,5.0,9.59,7.53,95.47,95.47,13.13,13.13,,,3.16,3.16,,,,,,Catálogo técnico Cintac,sí,,1.94
Cintac Ángulo Cintac 2.7,L (ángulo),100x100x6,100.0,100.0,6.0,6.0,11.41,8.96,112.52,112.52,15.58,15.58,,,3.14,3.14,,,,,,Catálogo técnico Cintac,sí,,1.92
I soldada serie IN (ICHA),I soldada,IN15 / 22.0 kgf/m,150.0,150.0,3.0,8.0,28.02,21.9957,1271.2725999999993,450.03015,169.50301333333326,60.00402,183.867,90.3015,6.735739423532305,4.007621887176753,5.247800000000001,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN15 / 17.4 kgf/m,150.0,150.0,3.0,6.0,22.14,17.3799,999.3617999999994,337.53105,133.24823999999992,45.00414,143.88299999999998,67.8105,6.71850375328665,3.9045236429107173,2.2896,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN15 / 15.1 kgf/m,150.0,150.0,3.0,5.0,19.2,15.072,857.35,281.2815,114.31333333333333,37.5042,123.45,56.565,6.682338350707283,3.8275420474502955,1.3805,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN15 / 15.7 kgf/m,150.0,100.0,3.0,8.0,20.02,15.7157,867.5659333333332,133.36348333333333,115.67545777777777,26.672696666666667,127.067,40.3015,6.582929568585836,2.580990634235749,3.541133333333334,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN15 / 12.7 kgf/m,150.0,100.0,3.0,6.0,16.14,12.6699,688.1417999999996,100.03105,91.75223999999996,20.00621,100.683,30.3105,6.529609426030438,2.489520165966407,1.5695999999999999,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN15 / 11.1 kgf/m,150.0,100.0,3.0,5.0,14.2,11.147,594.4333333333335,83.36483333333332,79.2577777777778,16.672966666666664,87.2,25.315,6.470046549092043,2.422965726294559,0.9638333333333332,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN15 / 9.2 kgf/m,150.0,75.0,3.0,5.0,11.7,9.1845,462.975,35.18775,61.730000000000004,9.3834,69.075,14.3775,6.29050974250202,1.7342145196024625,0.7555,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN15 / 8.1 kgf/m,150.0,75.0,3.0,4.0,10.26,8.0541,391.4022000000004,28.15695,52.18696000000005,7.50852,58.922999999999995,11.5695,6.176436073748311,1.6566055973777096,0.4514,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN20 / 43.2 kgf/m,200.0,200.0,4.0,12.0,55.04,43.2064,4428.765866666664,1600.0938666666666,442.8765866666664,160.00938666666667,482.17600000000004,240.704,8.970200450604823,5.391796812586418,23.441066666666664,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN20 / 37.1 kgf/m,200.0,200.0,4.0,10.0,47.2,37.052,3807.733333333334,1333.4293333333333,380.7733333333334,133.34293333333332,412.4,200.72,8.981776905723098,5.315131368924624,13.738666666666665,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN20 / 29.5 kgf/m,200.0,200.0,3.0,8.0,37.519999999999996,29.453199999999995,3106.5642666666713,1066.7080666666668,310.65642666666713,106.67080666666668,332.592,160.414,9.09931618758983,5.332015151154328,6.999466666666668,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN20 / 23.3 kgf/m,200.0,200.0,3.0,6.0,29.64,23.2674,2424.9967999999976,800.0423,242.49967999999976,80.00422999999999,259.308,120.423,9.045164827002228,5.195380682091434,3.0545999999999998,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN20 / 29.2 kgf/m,200.0,150.0,4.0,10.0,37.2,29.202,2904.4,562.596,290.44,75.0128,317.4,113.22,8.836021096466684,3.888900665624769,10.405333333333333,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN20 / 23.2 kgf/m,200.0,150.0,3.0,8.0,29.52,23.1732,2368.8576000000025,450.0414,236.88576000000026,60.005520000000004,255.79199999999997,90.414,8.958005004382208,3.9045236429107177,5.292800000000001,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN20 / 18.6 kgf/m,200.0,150.0,3.0,6.0,23.64,18.5574,1860.276799999998,337.5423,186.0276799999998,45.00564,201.108,67.923,8.870846183625117,3.7786821891096394,2.3346,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN20 / 16.2 kgf/m,200.0,150.0,3.0,5.0,20.7,16.2495,1597.725,281.29275,159.77249999999998,37.505700000000004,173.325,56.6775,8.785487044478277,3.6863290329446223,1.4255,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN20 / 16.9 kgf/m,200.0,100.0,3.0,8.0,21.52,16.8932,1631.1509333333356,133.37473333333335,163.11509333333356,26.67494666666667,178.992,40.414,8.706145901373926,2.4895201659664075,3.586133333333334,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN20 / 13.8 kgf/m,200.0,100.0,3.0,6.0,17.64,13.8474,1295.5567999999987,100.0423,129.55567999999988,20.00846,142.908,30.423,8.569962837641052,2.381455899139534,1.6145999999999998,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN20 / 12.3 kgf/m,200.0,100.0,3.0,5.0,15.7,12.3245,1122.3083333333338,83.37608333333333,112.23083333333338,16.675216666666664,124.575,25.4275,8.454857019411014,2.304469256957721,1.0088333333333332,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN25 / 57.1 kgf/m,250.0,200.0,4.0,16.0,72.72,57.0852,9119.9544,2133.4496000000004,729.596352,213.34496000000004,796.324,320.872,11.198746099069268,5.416444016749242,55.112533333333346,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN25 / 44.8 kgf/m,250.0,200.0,4.0,12.0,57.04,44.7764,7187.812533333335,1600.1205333333332,575.0250026666668,160.0120533333333,622.2760000000001,240.904,11.22557542899316,5.296470902456687,23.547733333333333,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN25 / 38.6 kgf/m,250.0,200.0,4.0,10.0,49.2,38.622,6168.900000000002,1333.456,493.51200000000017,133.3456,532.9,200.92,11.197506255477755,5.206031523880957,13.845333333333333,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN25 / 30.6 kgf/m,250.0,200.0,3.0,8.0,39.019999999999996,30.630699999999994,5007.149266666671,1066.7193166666668,400.57194133333365,106.67193166666668,428.267,160.5265,11.327958167953842,5.228552188451113,7.044466666666668,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN25 / 35.4 kgf/m,250.0,150.0,4.0,12.0,45.04,35.3564,5487.052533333333,675.1205333333334,438.9642026666666,90.01607111111112,479.47600000000006,135.904,11.037491144984301,3.8716087932176912,17.787733333333335,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN25 / 30.8 kgf/m,250.0,150.0,4.0,10.0,39.2,30.772,4728.066666666667,562.6226666666666,378.24533333333335,75.01635555555555,412.9,113.42,10.982438052556057,3.788485059706458,10.512,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN25 / 24.4 kgf/m,250.0,150.0,3.0,8.0,31.02,24.3507,3835.4426000000026,450.05265,306.8354080000002,60.007020000000004,331.467,90.5265,11.119540655852795,3.8089981777349795,5.337800000000001,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN25 / 19.7 kgf/m,250.0,150.0,3.0,6.0,25.14,19.7349,3016.6918,337.55355,241.335344,45.00714,262.08299999999997,68.0355,10.954254703165786,3.664280430587386,2.3796,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN25 / 22.9 kgf/m,250.0,100.0,4.0,10.0,29.200000000000003,22.922000000000004,3287.2333333333345,166.78933333333333,262.97866666666675,33.357866666666666,292.9,50.92,10.610206596398813,2.3899714371346854,7.1786666666666665,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN25 / 18.1 kgf/m,250.0,100.0,3.0,8.0,23.02,18.0707,2663.7359333333357,133.38598333333334,213.09887466666686,26.677196666666667,234.66699999999997,40.5265,10.75704350370637,2.407146030622684,3.631133333333334,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN25 / 15.0 kgf/m,250.0,100.0,3.0,6.0,19.14,15.0249,2123.4718000000007,100.05355,169.87774400000006,20.01071,188.88299999999998,30.5355,10.5330048028869,2.2863635324936458,1.6596,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN25 / 13.5 kgf/m,250.0,100.0,3.0,5.0,17.2,13.502,1846.4333333333338,83.38733333333333,147.71466666666672,16.677466666666668,165.7,25.54,10.361021918411257,2.201840315552833,1.0538333333333332,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN30 / 60.8 kgf/m,300.0,200.0,5.0,16.0,77.4,60.759,13720.647999999997,2133.6125,914.7098666666665,213.36125,998.5799999999999,321.675,13.314253790775055,5.250338367341311,55.796666666666674,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN30 / 46.3 kgf/m,300.0,200.0,4.0,12.0,59.04,46.3464,10659.859199999994,1600.1472,710.6572799999996,160.01472,767.3760000000001,241.104,13.4370075065733,5.206031523880957,23.6544,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN30 / 40.2 kgf/m,300.0,200.0,4.0,10.0,51.2,40.192,9145.06666666667,1333.4826666666665,609.6711111111113,133.34826666666666,658.4,201.12,13.364676701414568,5.103389396600393,13.951999999999998,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN30 / 34.0 kgf/m,300.0,200.0,4.0,8.0,43.36,34.0376,7586.370133333334,1066.8181333333334,505.7580088888889,106.68181333333334,547.856,161.136,13.22733574171716,4.960215644242937,7.449600000000001,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN30 / 48.2 kgf/m,300.0,150.0,5.0,16.0,61.4,48.199,10490.994666666662,900.2791666666667,699.3996444444441,120.03722222222223,771.3799999999999,181.675,13.071461483717265,3.829167944137245,42.14333333333334,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN30 / 36.9 kgf/m,300.0,150.0,4.0,12.0,47.04,36.9264,8170.099199999995,675.1472,544.6732799999996,90.01962666666667,594.576,136.104,13.178925663067265,3.7884850597064585,17.8944,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN30 / 32.3 kgf/m,300.0,150.0,4.0,10.0,41.2,32.342,7041.733333333334,562.6493333333333,469.4488888888889,75.0199111111111,513.4,113.62,13.07347916987586,3.695475235585875,10.618666666666666,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN30 / 27.8 kgf/m,300.0,150.0,4.0,8.0,35.36,27.7576,5880.663466666672,450.1514666666667,392.0442311111115,60.02019555555556,431.056,91.136,12.896059706661095,3.567986533532474,5.742933333333334,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN30 / 23.2 kgf/m,300.0,150.0,4.0,6.0,29.520000000000003,23.1732,4686.422399999992,337.6536,312.4281599999995,45.02048,347.544,68.652,12.599767709821602,3.3820304672342636,2.7872000000000003,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN30 / 24.5 kgf/m,300.0,100.0,4.0,10.0,31.200000000000003,24.492000000000004,4938.4000000000015,166.816,329.22666666666674,33.3632,368.4,51.12,12.58101948500404,2.3122860261366167,7.285333333333333,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN30 / 21.5 kgf/m,300.0,100.0,4.0,8.0,27.36,21.4776,4174.9568,133.4848,278.3304533333333,26.69696,314.256,41.136,12.352872147496615,2.208807463170279,4.036266666666667,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN30 / 18.5 kgf/m,300.0,100.0,4.0,6.0,23.520000000000003,18.463200000000004,3389.7023999999988,100.1536,225.98015999999993,20.03072,259.344,31.152,12.004998958767132,2.0635482287838602,2.0672,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN30 / 17.0 kgf/m,300.0,100.0,4.0,5.0,21.6,16.956,2988.799999999999,83.488,199.25333333333325,16.6976,231.60000000000002,26.16,11.763093571436482,1.9660074224644182,1.4626666666666668,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN35 / 62.7 kgf/m,350.0,200.0,5.0,16.0,79.9,62.7215,19202.506333333327,2133.6645833333337,1097.2860761904758,213.36645833333336,1195.205,321.9875,15.502636674013704,5.167609459515518,56.00500000000001,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN35 / 50.5 kgf/m,350.0,200.0,5.0,12.0,64.3,50.4755,15158.622333333327,1600.3395833333334,866.2069904761902,160.03395833333335,944.045,242.0375,15.354100537569146,4.988851620092456,24.44833333333333,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN35 / 41.8 kgf/m,350.0,200.0,4.0,10.0,53.2,41.762,12761.233333333328,1333.5093333333332,729.213333333333,133.35093333333333,788.9,201.32,15.487827704928272,5.006592145589722,14.058666666666666,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN35 / 35.6 kgf/m,350.0,200.0,4.0,8.0,45.36,35.6076,10600.8168,1066.8448,605.7609600000001,106.68448000000001,658.7560000000001,161.336,15.287382839259116,4.849691348203462,7.5562666666666685,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN35 / 50.2 kgf/m,350.0,150.0,5.0,16.0,63.9,50.1615,14736.853,900.33125,842.1058857142857,120.04416666666665,928.0049999999999,181.9875,15.186299010005145,3.7536237056100448,42.351666666666674,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN35 / 41.1 kgf/m,350.0,150.0,5.0,12.0,52.3,41.0555,11729.862333333333,675.3395833333333,670.2778476190475,90.04527777777777,741.245,137.0375,14.975992472679277,3.5934388460914577,18.688333333333333,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN35 / 33.9 kgf/m,350.0,150.0,4.0,10.0,43.2,33.912,9870.4,562.676,564.0228571428571,75.02346666666668,618.9,113.82,15.115603907270177,3.609003658547246,10.725333333333333,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN35 / 29.3 kgf/m,350.0,150.0,4.0,8.0,37.36,29.3276,8261.110133333335,450.17813333333334,472.0634361904763,60.02375111111111,521.956,91.336,14.870164477148354,3.4712729513719434,5.8496000000000015,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN35 / 24.7 kgf/m,350.0,150.0,4.0,6.0,31.52,24.7432,6612.80906666667,337.6802666666667,377.874803809524,45.02403555555556,423.84399999999994,68.852,14.4843792398147,3.273103485956415,2.893866666666667,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN35 / 31.6 kgf/m,350.0,100.0,5.0,12.0,40.3,31.6355,8301.10233333333,200.33958333333334,474.3487047619046,40.06791666666667,538.4449999999999,62.0375,14.352096995632484,2.229620044260918,12.928333333333333,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN35 / 26.1 kgf/m,350.0,100.0,4.0,10.0,33.2,26.062,6979.566666666664,166.84266666666664,398.8323809523808,33.36853333333333,448.9,51.32,14.499238312635125,2.241736274878117,7.3919999999999995,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN35 / 23.0 kgf/m,350.0,100.0,4.0,8.0,29.36,23.0476,5921.403466666668,133.51146666666668,338.3659123809524,26.702293333333337,385.156,41.336,14.20150271564174,2.132461788365133,4.1429333333333345,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN35 / 20.0 kgf/m,350.0,100.0,4.0,6.0,25.52,20.0332,4837.5890666666655,100.18026666666667,276.4336609523809,20.036053333333335,320.64399999999995,31.352,13.768104448488659,1.981302359222965,2.1738666666666666,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN40 / 67.6 kgf/m,400.0,200.0,6.0,16.0,86.08,67.5728,26098.41493333337,2133.9957333333336,1304.9207466666685,213.39957333333336,1431.936,323.312,17.412291802747852,4.979040331932815,57.378133333333345,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN40 / 52.4 kgf/m,400.0,200.0,5.0,12.0,66.8,52.438,20285.930666666645,1600.3916666666667,1014.2965333333323,160.03916666666666,1107.92,242.35,17.42646254425685,4.894686821629789,24.656666666666666,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN40 / 46.3 kgf/m,400.0,200.0,5.0,10.0,59.0,46.315,17499.666666666675,1333.7291666666665,874.9833333333338,133.37291666666664,960.5,202.375,17.222210072287805,4.754532479229182,14.958333333333332,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN40 / 37.2 kgf/m,400.0,200.0,4.0,8.0,47.36,37.1776,14182.26346666666,1066.8714666666667,709.113173333333,106.68714666666668,774.6560000000001,161.536,17.304813682226584,4.7462455527339555,7.662933333333335,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN40 / 55.0 kgf/m,400.0,150.0,6.0,16.0,70.08,55.0128,20196.76160000002,900.6624,1009.838080000001,120.08832000000001,1124.7359999999999,183.312,16.976330554238107,3.5849571557020288,43.72480000000001,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN40 / 43.0 kgf/m,400.0,150.0,5.0,12.0,54.8,43.018,15768.17066666665,675.3916666666667,788.4085333333326,90.05222222222223,875.12,137.35,16.96291073585554,3.5106502887815036,18.89666666666667,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN40 / 38.5 kgf/m,400.0,150.0,5.0,10.0,49.0,38.465,13696.333333333334,562.8958333333334,684.8166666666667,75.05277777777778,765.5,114.875,16.718762119329323,3.389346554725145,11.625,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN40 / 30.9 kgf/m,400.0,150.0,4.0,8.0,39.36,30.8976,11108.55680000001,450.2048,555.4278400000005,60.02730666666666,617.856,91.536,16.79969027976216,3.3820304672342636,5.956266666666668,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN40 / 26.3 kgf/m,400.0,150.0,4.0,6.0,33.519999999999996,26.313199999999995,8933.19573333335,337.7069333333333,446.6597866666675,45.02759111111111,505.14399999999995,69.052,16.32493346586981,3.1740808402730867,3.0005333333333337,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN40 / 33.6 kgf/m,400.0,100.0,5.0,12.0,42.8,33.598,11250.410666666656,200.39166666666668,562.5205333333328,40.07833333333333,642.3199999999999,62.35,16.21295970220415,2.163804123899472,13.136666666666667,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN40 / 30.6 kgf/m,400.0,100.0,5.0,10.0,39.0,30.615,9893.000000000005,167.0625,494.65000000000026,33.4125,570.5,52.375,15.926916420533725,2.0696989747675496,8.291666666666666,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN40 / 24.6 kgf/m,400.0,100.0,4.0,8.0,31.36,24.6176,8034.850133333331,133.53813333333335,401.74250666666654,26.70762666666667,461.05600000000004,41.536,16.006665278356177,2.0635482287838602,4.249600000000001,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN40 / 21.6 kgf/m,400.0,100.0,4.0,6.0,27.52,21.6032,6604.47573333335,100.20693333333334,330.2237866666675,20.041386666666668,386.94399999999996,31.552,15.491553087223506,1.9082034247106674,2.2805333333333335,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN45 / 69.9 kgf/m,450.0,200.0,6.0,16.0,89.08,69.9278,33802.344933333356,2134.0857333333333,1502.3264414814826,213.40857333333332,1650.886,323.762,19.479746926956544,4.894584460642081,57.738133333333344,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN45 / 57.7 kgf/m,450.0,200.0,6.0,12.0,73.56,57.7446,26892.47879999998,1600.7668,1195.221279999999,160.07668,1323.4139999999998,243.834,19.120293618528123,4.664908975284401,26.1936,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN45 / 48.3 kgf/m,450.0,200.0,5.0,10.0,61.5,48.2775,22676.12500000001,1333.78125,1007.8277777777782,133.378125,1111.125,202.6875,19.202017593857082,4.656984002549289,15.166666666666666,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN45 / 42.2 kgf/m,450.0,200.0,5.0,8.0,53.7,42.1545,19036.930999999993,1067.11875,846.085822222222,106.711875,942.645,162.7125,18.82830934456607,4.457786170499819,8.668333333333335,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN45 / 57.4 kgf/m,450.0,150.0,6.0,16.0,73.08,57.3678,26264.69160000002,900.7524,1167.3196266666675,120.10032,1303.686,183.762,18.95775400150581,3.510778617694806,44.08480000000001,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN45 / 48.3 kgf/m,450.0,150.0,6.0,12.0,61.56,48.3246,21135.718799999984,675.7668,939.3652799999993,90.10224,1060.614,138.834,18.529308221244918,3.3132111947554193,20.433600000000002,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN45 / 40.4 kgf/m,450.0,150.0,5.0,10.0,51.5,40.4275,17835.291666666668,562.9479166666666,792.6796296296296,75.05972222222222,891.125,115.1875,18.609576648652467,3.3062104452213283,11.833333333333334,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN45 / 35.9 kgf/m,450.0,150.0,5.0,8.0,45.7,35.8745,15129.224333333328,450.45208333333335,672.4099703703702,60.06027777777778,765.845,92.7125,18.19492338684738,3.13954129575888,6.961666666666668,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN45 / 38.9 kgf/m,450.0,100.0,6.0,12.0,49.56,38.9046,15378.95879999999,200.7668,683.5092799999995,40.15336,797.8139999999999,63.834,17.615615321290925,2.0127058068809913,14.673599999999999,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN45 / 32.6 kgf/m,450.0,100.0,5.0,10.0,41.5,32.5775,12994.458333333338,167.11458333333331,577.5314814814817,33.422916666666666,671.125,52.6875,17.695182336207974,2.006703124460336,8.5,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN45 / 29.6 kgf/m,450.0,100.0,5.0,8.0,37.7,29.5945,11221.517666666663,133.78541666666666,498.73411851851836,26.757083333333334,589.0450000000001,42.7125,17.25262265586734,1.8837953159033163,5.255000000000001,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN45 / 26.6 kgf/m,450.0,100.0,5.0,6.0,33.9,26.6115,9415.593000000003,100.45625,418.4708000000001,20.09125,506.2049999999999,32.7375,16.665716492089093,1.7214270851359137,3.29,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN50 / 72.3 kgf/m,500.0,200.0,6.0,16.0,92.08,72.2828,42619.77493333337,2134.1757333333335,1704.7909973333349,213.41757333333334,1877.3359999999998,324.212,21.51408700741274,4.814292061245368,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN50 / 60.1 kgf/m,500.0,200.0,6.0,12.0,76.56,60.0996,33975.54880000001,1600.8568,1359.0219520000005,160.08568,1511.0639999999999,244.284,21.0660096057738,4.5727270649872915,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN50 / 50.2 kgf/m,500.0,200.0,5.0,10.0,64.0,50.24,28621.333333333343,1333.8333333333333,1144.8533333333337,133.38333333333333,1268.0,203.0,21.147300852197034,4.56521038215473,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN50 / 44.1 kgf/m,500.0,200.0,5.0,8.0,56.2,44.117,24090.989333333328,1067.1708333333333,963.6395733333331,106.71708333333333,1080.02,163.025,20.70423364021733,4.357614584102459,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN50 / 59.7 kgf/m,500.0,150.0,6.0,16.0,76.08,59.7228,33246.12160000002,900.8424,1329.8448640000008,120.11232,1490.136,184.212,20.90428138016303,3.441035534842742,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN50 / 50.7 kgf/m,500.0,150.0,6.0,12.0,64.56,50.6796,26829.7888,675.8568,1073.191552,90.11424000000001,1218.2640000000001,139.284,20.385758082294583,3.235531132602061,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN50 / 42.4 kgf/m,500.0,150.0,5.0,10.0,54.0,42.39,22618.0,563.0,904.72,75.06666666666666,1023.0,115.5,20.46587041520228,3.22892024149342,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN50 / 37.8 kgf/m,500.0,150.0,5.0,8.0,48.2,37.837,19249.28266666669,450.50416666666666,769.9713066666677,60.06722222222222,883.22,93.025,19.98406142221082,3.0572143324294703,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN55 / 74.6 kgf/m,550.0,200.0,6.0,16.0,95.08,74.6378,52588.20493333337,2134.2657333333336,1912.2983612121225,213.42657333333335,2111.286,324.662,23.517956720740205,4.737832026025125,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN55 / 62.5 kgf/m,550.0,200.0,6.0,12.0,79.56,62.4546,42015.61880000001,1600.9468,1527.840683636364,160.09467999999998,1706.214,244.734,22.980421750067173,4.485811944162522,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN55 / 52.2 kgf/m,550.0,200.0,5.0,10.0,66.5,52.2025,35366.54166666668,1333.8854166666665,1286.056060606061,133.38854166666664,1431.125,203.3125,23.061389685697375,4.478663563865424,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN55 / 46.1 kgf/m,550.0,200.0,5.0,8.0,58.7,46.0795,29847.54766666669,1067.2229166666668,1085.3653696969707,106.72229166666668,1223.645,163.3375,22.549414851935282,4.263914823203479,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN55 / 62.1 kgf/m,550.0,150.0,6.0,16.0,79.08,62.0778,41178.55159999999,900.9324,1497.4018763636361,120.12432,1684.0859999999998,184.662,22.819293964448438,3.3753030550162193,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN55 / 53.0 kgf/m,550.0,150.0,6.0,12.0,67.56,53.0346,33330.8588,675.9468,1212.0312290909092,90.12624000000001,1383.414,139.734,22.211527765107782,3.163089188581326,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN55 / 44.4 kgf/m,550.0,150.0,5.0,10.0,56.5,44.3525,28075.708333333332,563.0520833333334,1020.9348484848484,75.07361111111112,1161.125,115.8125,22.29159464328232,3.156821755946962,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN55 / 39.8 kgf/m,550.0,150.0,5.0,8.0,50.7,39.7995,23971.840999999997,450.55625,871.703309090909,60.07416666666666,1006.845,93.3375,21.74436425205019,2.9810587121633296,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN60 / 77.0 kgf/m,600.0,200.0,6.0,16.0,98.08,76.9928,63745.13493333334,2134.3557333333333,2124.8378311111114,213.43557333333334,2352.736,325.112,25.493724824704177,4.664908975284401,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN60 / 64.8 kgf/m,600.0,200.0,6.0,12.0,82.56,64.8096,51050.18879999998,1601.0368,1701.6729599999994,160.10368,1908.8639999999998,245.184,24.866452604187007,4.403680631105666,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN60 / 54.2 kgf/m,600.0,200.0,5.0,10.0,69.0,54.165,42942.99999999998,1333.9375,1431.4333333333327,133.39375,1600.5,203.625,24.947190600157352,4.396865649099581,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN60 / 48.0 kgf/m,600.0,200.0,5.0,8.0,61.2,48.042,36337.85599999999,1067.275,1211.2618666666665,106.7275,1373.52,163.65,24.367105223917427,4.176018916016558,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN60 / 64.4 kgf/m,600.0,150.0,6.0,16.0,82.08,64.4328,50099.48160000005,901.0224,1669.9827200000016,120.13632,1885.5359999999998,185.112,24.705744294993245,3.3132111947554193,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN60 / 55.4 kgf/m,600.0,150.0,6.0,12.0,70.56,55.3896,40676.42879999994,676.0368,1355.8809599999981,90.13824,1556.0639999999999,140.184,24.009997917534253,3.09532234317579,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN60 / 46.3 kgf/m,600.0,150.0,5.0,10.0,59.0,46.315,34239.666666666664,563.1041666666666,1141.322222222222,75.08055555555555,1305.5,116.125,24.090108620206205,3.08935890082054,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie IN (ICHA),I soldada,IN60 / 41.8 kgf/m,600.0,150.0,5.0,8.0,53.2,41.762,29328.149333333327,450.60833333333335,977.6049777777775,60.08111111111111,1136.72,93.65,23.47937398028886,2.910340436037352,,,,,,CALCULO_VIGAS_DOBLE_TE_2 / Propiedades para el diseño,sí,,
I soldada serie HN (ICHA),I soldada,HN 50 / 462 kgf/m,500.0,500.0,22.0,50.0,588.0,462.0,266000.0,104000.0,10600.0,4170.0,12130.0,,21.3,13.3,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 50 / 380 kgf/m,500.0,500.0,20.0,40.0,484.0,380.0,224000.0,83400.0,8980.0,3330.0,10082.0,,21.5,13.1,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 50 / 336 kgf/m,500.0,500.0,18.0,35.0,427.0,336.0,201000.0,72900.0,8060.0,2920.0,8969.5,,21.7,13.1,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 50 / 306 kgf/m,500.0,500.0,16.0,32.0,390.0,306.0,187000.0,66700.0,7460.0,2670.0,8248.4,,21.9,13.1,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 50 / 269 kgf/m,500.0,500.0,14.0,28.0,342.0,269.0,166000.0,58300.0,6650.0,2330.0,7298.0,,22.0,13.1,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 50 / 246 kgf/m,500.0,500.0,14.0,25.0,313.0,246.0,152000.0,52100.0,6070.0,2080.0,6646.2,,22.0,12.9,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 50 / 223 kgf/m,500.0,500.0,14.0,22.0,284.0,223.0,137000.0,45800.0,5470.0,1830.0,5985.8,,22.0,12.7,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 50 / 208 kgf/m,500.0,500.0,14.0,20.0,264.0,208.0,127000.0,41700.0,5060.0,1670.0,5540.6,,21.9,12.6,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 50 / 192 kgf/m,500.0,500.0,14.0,18.0,245.0,192.0,116000.0,37500.0,4650.0,1500.0,5091.5,,21.8,12.4,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 50 / 177 kgf/m,500.0,500.0,14.0,16.0,226.0,177.0,106000.0,33300.0,4230.0,1330.0,4638.6,,21.6,12.2,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 45 / 341 kgf/m,450.0,450.0,20.0,40.0,434.0,341.0,160000.0,60800.0,7120.0,2700.0,8064.5,,19.2,11.8,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 45 / 301 kgf/m,450.0,450.0,18.0,35.0,383.0,301.0,144000.0,53200.0,6410.0,2360.0,7186.1,,19.4,11.8,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 45 / 275 kgf/m,450.0,450.0,16.0,32.0,350.0,275.0,134000.0,48600.0,5940.0,2160.0,6615.2,,19.6,11.8,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 45 / 241 kgf/m,450.0,450.0,14.0,28.0,307.0,241.0,119000.0,42500.0,5310.0,1890.0,5860.5,,19.7,11.8,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 45 / 214 kgf/m,450.0,450.0,12.0,25.0,273.0,214.0,108000.0,38000.0,4810.0,1690.0,5261.2,,19.9,11.8,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 45 / 194 kgf/m,450.0,450.0,12.0,22.0,247.0,194.0,97400.0,33400.0,4330.0,1490.0,4731.7,,19.9,11.6,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 45 / 180 kgf/m,450.0,450.0,12.0,20.0,229.0,180.0,90200.0,30400.0,4010.0,1350.0,4374.3,,19.8,11.5,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 45 / 166 kgf/m,450.0,450.0,12.0,18.0,212.0,166.0,82700.0,27300.0,3680.0,1220.0,4013.4,,19.8,11.4,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 45 / 152 kgf/m,450.0,450.0,12.0,16.0,194.0,152.0,75100.0,24300.0,3340.0,1080.0,3649.0,,19.7,11.2,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 45 / 139 kgf/m,450.0,450.0,12.0,14.0,177.0,139.0,67400.0,21300.0,3000.0,945.0,3281.1,,19.5,11.0,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 40 / 301 kgf/m,400.0,400.0,20.0,40.0,384.0,301.0,110000.0,42700.0,5480.0,2130.0,6272.0,,16.9,10.5,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 40 / 266 kgf/m,400.0,400.0,18.0,35.0,339.0,266.0,98900.0,37300.0,4950.0,1870.0,5600.1,,17.1,10.5,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 40 / 243 kgf/m,400.0,400.0,16.0,32.0,310.0,243.0,91900.0,34100.0,4600.0,1710.0,5162.0,,17.2,10.5,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 40 / 214 kgf/m,400.0,400.0,14.0,28.0,272.0,214.0,82400.0,29900.0,4120.0,1490.0,4580.6,,17.4,10.5,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 40 / 190 kgf/m,400.0,400.0,12.0,25.0,242.0,190.0,74700.0,26700.0,3740.0,1330.0,4117.5,,17.6,10.5,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 40 / 172 kgf/m,400.0,400.0,12.0,22.0,219.0,172.0,67500.0,23500.0,3370.0,1170.0,3706.6,,17.6,10.4,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 40 / 160 kgf/m,400.0,400.0,12.0,20.0,203.0,160.0,62500.0,21300.0,3120.0,1070.0,3428.8,,17.5,10.2,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 40 / 147 kgf/m,400.0,400.0,12.0,18.0,188.0,147.0,57400.0,19200.0,2870.0,960.0,3147.9,,17.5,10.1,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 40 / 135 kgf/m,400.0,400.0,12.0,16.0,172.0,135.0,52200.0,17100.0,2610.0,854.0,2863.9,,17.4,9.96,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 40 / 123 kgf/m,400.0,400.0,12.0,14.0,157.0,123.0,46900.0,14900.0,2340.0,747.0,2576.8,,17.3,9.77,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 35 / 232 kgf/m,350.0,350.0,18.0,35.0,295.0,232.0,64300.0,25000.0,3680.0,1430.0,4211.6,,14.8,9.2,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 35 / 212 kgf/m,350.0,350.0,16.0,32.0,270.0,212.0,59900.0,22900.0,3430.0,1310.0,3888.8,,14.9,9.21,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 35 / 186 kgf/m,350.0,350.0,14.0,28.0,237.0,186.0,53900.0,20000.0,3080.0,1140.0,3458.1,,15.1,9.19,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 35 / 166 kgf/m,350.0,350.0,12.0,25.0,211.0,166.0,49000.0,17900.0,2800.0,1020.0,3113.8,,15.2,9.2,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 35 / 150 kgf/m,350.0,350.0,12.0,22.0,191.0,150.0,44300.0,15700.0,2530.0,899.0,2806.5,,15.2,9.08,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 35 / 134 kgf/m,350.0,350.0,10.0,20.0,171.0,134.0,40600.0,14300.0,2320.0,817.0,2550.2,,15.4,9.14,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 35 / 124 kgf/m,350.0,350.0,10.0,18.0,157.0,124.0,37300.0,12900.0,2130.0,735.0,2338.1,,15.4,9.04,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 35 / 113 kgf/m,350.0,350.0,10.0,16.0,144.0,113.0,33900.0,11400.0,1940.0,653.0,2123.2,,15.4,8.92,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 35 / 102 kgf/m,350.0,350.0,10.0,14.0,130.0,102.0,30500.0,10000.0,1740.0,572.0,1905.6,,15.3,8.77,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 35 / 91 kgf/m,350.0,350.0,10.0,12.0,117.0,91.5,26900.0,8580.0,1540.0,490.0,1685.3,,15.2,8.58,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 30 / 180 kgf/m,300.0,300.0,16.0,32.0,230.0,180.0,36400.0,14400.0,2430.0,961.0,2795.6,,12.6,7.92,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 30 / 159 kgf/m,300.0,300.0,14.0,28.0,202.0,159.0,32900.0,12600.0,2190.0,840.0,2493.2,,12.8,7.9,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 30 / 141 kgf/m,300.0,300.0,12.0,25.0,180.0,141.0,30000.0,11300.0,2000.0,750.0,2250.0,,12.9,7.91,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 30 / 128 kgf/m,300.0,300.0,12.0,22.0,163.0,128.0,27200.0,9900.0,1820.0,660.0,2031.4,,12.9,7.8,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 30 / 115 kgf/m,300.0,300.0,10.0,20.0,146.0,115.0,25000.0,9000.0,1670.0,600.0,1849.0,,13.1,7.85,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 30 / 106 kgf/m,300.0,300.0,10.0,18.0,134.0,106.0,23000.0,8100.0,1540.0,540.0,1697.0,,13.1,7.76,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 30 / 92 kgf/m,300.0,300.0,8.0,16.0,117.0,92.2,20700.0,7200.0,1380.0,480.0,1506.8,,13.3,7.83,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 30 / 83 kgf/m,300.0,300.0,8.0,14.0,106.0,83.0,18500.0,6300.0,1240.0,420.0,1349.2,,13.2,7.72,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 30 / 73 kgf/m,300.0,300.0,8.0,12.0,94.1,73.9,16300.0,5400.0,1090.0,360.0,1189.2,,13.2,7.58,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 30 / 64 kgf/m,300.0,300.0,8.0,10.0,82.4,64.7,14100.0,4500.0,939.0,300.0,1026.8,,13.1,7.39,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 25 / 131 kgf/m,250.0,250.0,14.0,28.0,167.0,131.0,18200.0,7300.0,1460.0,584.0,1685.7,,10.4,6.61,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 25 / 117 kgf/m,250.0,250.0,12.0,25.0,149.0,117.0,16700.0,6510.0,1330.0,521.0,1526.2,,10.6,6.61,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 25 / 106 kgf/m,250.0,250.0,12.0,22.0,135.0,106.0,15200.0,5730.0,1220.0,459.0,1381.3,,10.6,6.52,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 25 / 95 kgf/m,250.0,250.0,10.0,20.0,121.0,95.0,14000.0,5210.0,1120.0,417.0,1260.2,,10.8,6.56,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 25 / 87 kgf/m,250.0,250.0,10.0,18.0,111.0,87.4,13000.0,4690.0,1040.0,375.0,1158.5,,10.8,6.49,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 25 / 76 kgf/m,250.0,250.0,8.0,16.0,97.4,76.5,11700.0,4170.0,933.0,333.0,1031.0,,10.9,6.54,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 25 / 68 kgf/m,250.0,250.0,8.0,14.0,87.8,68.9,10500.0,3650.0,839.0,292.0,924.6,,10.9,6.45,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 25 / 57 kgf/m,250.0,250.0,6.0,12.0,73.6,57.7,9080.0,3130.0,726.0,250.0,790.6,,11.1,6.52,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 25 / 50 kgf/m,250.0,250.0,6.0,10.0,63.8,50.1,7810.0,2600.0,625.0,208.0,679.4,,11.1,6.39,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 25 / 42 kgf/m,250.0,250.0,6.0,8.0,54.0,42.4,6500.0,2080.0,520.0,167.0,566.1,,11.0,6.21,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 20 / 90 kgf/m,200.0,200.0,10.0,25.0,115.0,90.3,7990.0,3330.0,799.0,333.0,931.2,,8.34,5.38,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 20 / 81 kgf/m,200.0,200.0,10.0,22.0,104.0,81.3,7320.0,2930.0,732.0,293.0,844.0,,8.41,5.32,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 20 / 75 kgf/m,200.0,200.0,10.0,20.0,96.0,75.4,6850.0,2670.0,685.0,267.0,784.0,,8.45,5.27,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 20 / 69 kgf/m,200.0,200.0,10.0,18.0,88.4,69.4,6350.0,2400.0,635.0,240.0,722.4,,8.47,5.21,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 20 / 60 kgf/m,200.0,200.0,8.0,16.0,77.4,60.8,5750.0,2130.0,575.0,213.0,645.2,,8.61,5.25,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 20 / 54 kgf/m,200.0,200.0,8.0,14.0,69.8,54.8,5190.0,1870.0,519.0,187.0,580.0,,8.63,5.17,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 20 / 46 kgf/m,200.0,200.0,6.0,12.0,58.6,46.0,4520.0,1600.0,452.0,160.0,497.7,,8.79,5.23,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 20 / 39 kgf/m,200.0,200.0,6.0,10.0,50.8,39.9,3900.0,1330.0,390.0,133.0,428.6,,8.77,5.12,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
I soldada serie HN (ICHA),I soldada,HN 20 / 33 kgf/m,200.0,200.0,6.0,8.0,43.0,33.8,3260.0,1070.0,326.0,107.0,358.0,,8.71,4.98,,,,,,880734b9-MANUAL-ICHA / Serie HN (Diseño),sí,aprox. (B·tf·(H−tf)+tw·(H−2tf)²/4),
'''

def find_db():
    if BASE_DB: return BASE_DB
    hits = glob.glob(f'{INPUT_DIR}/**/base_datos_perfiles.csv', recursive=True)
    return hits[0] if len(hits) == 1 else None

_db_path = find_db()
db_raw = pd.read_csv(_db_path) if _db_path else pd.read_csv(io.StringIO(DB_EMBEBIDA))
print('Base de datos de perfiles:', _db_path or 'incrustada en el notebook', '|', len(db_raw), 'filas')
def fl(v):
    return float(v) if pd.notna(v) and v != '' else np.nan

def props(r):
    """Propiedades en mm, N. tipo_kind: I | RECT | ROUND | ANG | OTHER."""
    t = r['tipo']
    A = fl(r['A_cm2']) * 100
    kind = {'I soldada': 'I', 'I/H ala ancha': 'I', 'I': 'I', 'H': 'I', 'I inclinada': 'I',
            'HSS rectangular': 'RECT', 'HSS cuadrado': 'RECT', 'HSS circular': 'ROUND',
            'L (ángulo)': 'ANG'}.get(t, 'OTHER')
    d = dict(kind=kind, A=A, peso=fl(r['peso_kgf_m']))
    if kind == 'I':
        H, B, tw, tf = fl(r['H_mm']), fl(r['B_mm']), fl(r['tw_mm']), fl(r['tf_mm'])
        d.update(H=H, B=B, tw=tw, tf=tf,
                 Ix=fl(r['Ix']) * 1e4, Iy=fl(r['Iy']) * 1e4, Sx=fl(r['Wx']) * 1e3, Sy=fl(r['Wy']) * 1e3,
                 Zx=fl(r['Zx']) * 1e3, Zy=fl(r['Zy']) * 1e3, J=fl(r['J']) * 1e4 if pd.notna(r['J']) else np.nan)
        d['Cw'] = d['Iy'] * (H - tf) ** 2 / 4
    elif kind == 'RECT':
        H, B, t = fl(r['H_mm']), fl(r['B_mm']), fl(r['tw_mm'])
        d.update(H=H, B=B, tw=t, tf=t, Ix=fl(r['Ix']) * 1e4, Iy=fl(r['Iy']) * 1e4,
                 Sx=fl(r['Wx']) * 1e3, Sy=fl(r['Wy']) * 1e3)
        d['Zx'] = d['Sx'] * 1.12; d['Zy'] = d['Sy'] * 1.12        # aprox. para tubo rectangular (no viene en la planilla)
        Am = (B - t) * (H - t); p = 2 * ((B - t) + (H - t))
        d['J'] = 4 * Am ** 2 * t / p                               # torsión de sección cerrada de pared delgada
        d['Cw'] = 0.0
    elif kind == 'ROUND':
        D, t = fl(r['H_mm']), fl(r['tw_mm'])
        I = fl(r['Ix']) * 1e4
        d.update(H=D, B=D, tw=t, tf=t, Ix=I, Iy=I, Sx=fl(r['Wx']) * 1e3, Sy=fl(r['Wx']) * 1e3,
                 Zx=fl(r['Wx']) * 1e3 * 4 / 3 if pd.notna(r['Wx']) else np.nan)
        d['Zy'] = d['Zx']; d['J'] = 2 * I; d['Cw'] = 0.0
    elif kind == 'ANG':
        leg, t = fl(r['H_mm']), fl(r['tw_mm'])
        d.update(H=leg, B=fl(r['B_mm']), tw=t, tf=t, Ix=fl(r['Ix']) * 1e4, Iy=fl(r['Iy']) * 1e4,
                 Sx=fl(r['Wx']) * 1e3, Sy=fl(r['Wy']) * 1e3)
        d['rx'] = fl(r['ix']) * 10
        d['ry'] = fl(r['r_min_cm']) * 10 if pd.notna(r['r_min_cm']) and r['r_min_cm'] != '' else np.nan
        # Z plástico no viene en la planilla de ángulos: se aproxima como 1,5·S (flexión de ángulos no es el control)
        d['Zx'] = 1.5 * d['Sx']; d['Zy'] = 1.5 * d['Sy']
    return d

cat = db_raw[db_raw['tipo'].isin(['I soldada', 'I/H ala ancha', 'I', 'H', 'I inclinada', 'HSS rectangular',
                                   'HSS cuadrado', 'HSS circular', 'L (ángulo)'])].copy()
cat = cat[cat['validado'].astype(str).isin(['sí', 'si', 'True', 'true'])]
cat = cat[cat['A_cm2'].notna() & cat['Ix'].notna()]
PROPS = []
for _, r in cat.iterrows():
    d = props(r)
    if d['kind'] != 'ANG':
        d['rx'] = np.sqrt(d['Ix'] / d['A'])
        d['ry'] = np.sqrt(d['Iy'] / d['A'])
    d.update(designacion=r['designacion'], tipo=r['tipo'], familia=r['familia'],
             laminado=str(r['familia']).startswith('Copromet') or str(r['familia']).startswith('Cintac'))
    PROPS.append(d)
PROPS = [p for p in PROPS if not (p['kind'] == 'ANG' and not np.isfinite(p['ry']))]
cand = pd.DataFrame(PROPS)
print('Perfiles candidatos (validados, tipos soportados):', len(cand))
print(cand.groupby('tipo').size().to_dict())

In [ ]:
E = E_MPA
FYE = FY_EFF = RY * FY_MPA if USE_RY else FY_MPA
LAM_ALAS_NO_ATIESADA = 0.40 * math.sqrt(E / FYE)     # Tabla 9, alas no atiesadas I/H (2025)
LAM_ALMA_ARR = 1.57 * math.sqrt(E / FYE)              # Tabla 9, almas I/H/C arriostramiento (2025)
LAM_ALMA_FLEX_A = lambda ca: 3.96 * math.sqrt(E / FYE) * (1 - 3.04 * ca)  # Tabla 9, almas flexo-compresión, Ca<=0.114 (2025)
LAM_HSS_WALL = 1.40 * math.sqrt(E / FYE)              # 2003 Tabla 8.1 rectangular espesor uniforme (completa 2025)
LAM_ANG = 0.45 * math.sqrt(E / FYE)                   # 2003 Tabla 8.1 puntales de ángulos (completa)
LAM_GLOBAL_ARR = 1.5 * math.pi * math.sqrt(E / FY_MPA)  # NCh2369:2025 8.8.4 (usa Fy, no Ry·Fy)

def fcr_comp(lam, A):
    """Pn = Fcr·A según AISC E3 (lam = KL/r máx)."""
    Fe = math.pi ** 2 * E / lam ** 2
    if lam <= 4.71 * math.sqrt(E / FY_MPA):
        Fcr = 0.658 ** (FY_MPA / Fe) * FY_MPA
    else:
        Fcr = 0.877 * Fe
    return Fcr * A

def Mn_major(c, Lb, Cb):
    """Momento nominal eje fuerte (F2 para I doblemente simétricas; Mp en tubos y ángulos)."""
    Mp = FY_MPA * c['Zx']
    if c['kind'] != 'I' or not np.isfinite(c['J']):
        return Mp
    ry = c['ry']; Sx = c['Sx']; ho = c['H'] - c['tf']; J = c['J']
    Lp = 1.76 * ry * math.sqrt(E / FY_MPA)
    rts = math.sqrt(math.sqrt(c['Iy'] * c['Cw']) / Sx)
    Lr = 1.95 * rts * E / (0.7 * FY_MPA) * math.sqrt(J / (Sx * ho)) * \
         math.sqrt(1 + math.sqrt(1 + 6.76 * (0.7 * FY_MPA * Sx * ho / (E * J)) ** 2))
    if Lb <= Lp:
        return Mp
    if Lb <= Lr:
        return min(Cb * (Mp - (Mp - 0.7 * FY_MPA * Sx) * (Lb - Lp) / (Lr - Lp)), Mp)
    Fcr = Cb * math.pi ** 2 * E / (Lb / rts) ** 2 * math.sqrt(1 + 0.078 * J / (Sx * ho) * (Lb / rts) ** 2)
    return min(Fcr * Sx, Mp)

def Mn_minor(c):
    return min(FY_MPA * c['Zy'], 1.6 * FY_MPA * c['Sy'])

def Vn(c, axis):
    if c['kind'] == 'I':
        Aw = c['H'] * c['tw'] if axis == 3 else 2 * c['B'] * c['tf']
    elif c['kind'] == 'RECT':
        Aw = 2 * c['H'] * c['tw'] if axis == 3 else 2 * c['B'] * c['tw']
    elif c['kind'] == 'ROUND':
        Aw = 0.5 * c['A']
    else:
        return np.nan
    return 0.6 * FY_MPA * Aw

def check_section(c, frames_df, dem_g):
    """Devuelve (UC máximo, detalle) para un perfil frente a todas las barras de un grupo."""
    tipo = frames_df['tipo_miembro'].iloc[0]
    if c['kind'] == 'OTHER':
        return np.inf, {'motivo': 'tipo no soportado'}
    if c['kind'] == 'ANG' and not np.isfinite(c['ry']):
        return np.inf, {'motivo': 'sin r_min'}
    Ca_max = 0.0
    if c['kind'] == 'I' and tipo != 'Arriostramiento':
        Ca_max = max(0.0, OMEGA_C * (-dem_g['P'].min()) / (FYE * c['A']))
    uc_all, g_all = 0.0, {}
    for _, fr_ in frames_df.iterrows():
        dfr = dem_g[dem_g['Frame'] == fr_['Frame']]
        if dfr.empty:
            continue
        L = fr_['L_mm']
        kx, ky = fr_['Kx'] * L, fr_['Ky'] * L
        rx = c['rx']; ry = c['ry']
        lam = max(kx / rx, ky / ry)
        Pn = fcr_comp(lam, c['A'])
        Pc = Pn / OMEGA_C
        Tc = FY_MPA * c['A'] / OMEGA_T
        Mn3 = Mn_major(c, L * fr_['Lb_factor'], fr_['Cb'])
        Mn2 = Mn_minor(c)
        M3c, M2c = Mn3 / OMEGA_B, Mn2 / OMEGA_B
        V3c, V2c = Vn(c, 3), Vn(c, 2)
        omv3 = OMEGA_V_ROLLED if (c['laminado'] and c['kind'] == 'I' and c['tipo'] != 'I soldada') else OMEGA_V
        V3c = V3c / omv3 if np.isfinite(V3c) else np.inf
        V2c = V2c / OMEGA_V if np.isfinite(V2c) else np.inf
        P = dfr['P'].values
        comp = np.maximum(-P, 0)
        tens = np.maximum(P, 0)
        ratio_P = np.where(P < 0, comp / Pc, tens / Tc)
        cap = np.where(P < 0, Pc, Tc)
        m3 = np.abs(dfr['M3'].values) / M3c
        m2 = np.abs(dfr['M2'].values) / M2c
        ratio = np.where(ratio_P >= 0.2, ratio_P + 8 / 9 * (m3 + m2), ratio_P / 2 + (m3 + m2))
        vr3 = np.abs(dfr['V3'].values) / V3c if np.isfinite(V3c) else np.zeros(len(P))
        vr2 = np.abs(dfr['V2'].values) / V2c if np.isfinite(V2c) else np.zeros(len(P))
        ratio = np.maximum(ratio, np.maximum(vr3, vr2))
        if not np.all(np.isfinite(ratio)):
            return np.inf, {'motivo': 'resultado no finito'}
        uc = float(ratio.max())
        if uc > uc_all:
            uc_all = uc
            i = int(np.argmax(ratio))
            g_all = dict(frame=fr_['Frame'], combo=dfr['combo'].iloc[i], lam=round(lam, 1), UC=round(uc, 3))
        g_all.setdefault('lam_max', 0.0)
        g_all['lam_max'] = max(g_all['lam_max'], round(lam, 1))
    return uc_all, g_all

def check_local(c, tipo, Ca_max):
    """Relaciones ancho/espesor frente a los límites de NCh2369. Devuelve (ok, detalle)."""
    sq = math.sqrt(E / FYE)
    if c['kind'] == 'I':
        lam_f = (c['B'] / 2) / c['tf']
        lim_f = LAM_ALAS_NO_ATIESADA
        h_clear = c['H'] - 2 * c['tf']
        lam_w = h_clear / c['tw']
        if tipo == 'Arriostramiento':
            lim_w = LAM_ALMA_ARR
        else:
            lim_w = LAM_ALMA_FLEX_A(Ca_max) if Ca_max <= 0.114 else 1.57 * sq     # Ca>0.114: piso conservador
        ok = lam_f <= lim_f and lam_w <= lim_w
        return ok, dict(lam_ala=round(lam_f, 2), lim_ala=round(lim_f, 2), lam_alma=round(lam_w, 2), lim_alma=round(lim_w, 2))
    if c['kind'] == 'RECT':
        lam_b = (c['B'] - 2 * c['tw']) / c['tw']; lam_h = (c['H'] - 2 * c['tw']) / c['tw']
        ok = lam_b <= LAM_HSS_WALL and lam_h <= LAM_HSS_WALL
        return ok, dict(lam_ala=round(lam_b, 2), lim_ala=round(LAM_HSS_WALL, 2), lam_alma=round(lam_h, 2), lim_alma=round(LAM_HSS_WALL, 2))
    if c['kind'] == 'ROUND':
        lam = c['H'] / c['tw']
        lim = (0.062 if tipo == 'Arriostramiento' else 0.077) * E / FYE
        return lam <= lim, dict(lam_ala=round(lam, 2), lim_ala=round(lim, 2), lam_alma=None, lim_alma=None)
    if c['kind'] == 'ANG':
        lam = c['H'] / c['tw']
        return lam <= LAM_ANG, dict(lam_ala=round(lam, 2), lim_ala=round(LAM_ANG, 2), lam_alma=None, lim_alma=None)
    return False, {}

In [ ]:
RESULT_COLS = ['grupo', 'tipo_miembro', 'designacion', 'familia', 'tipo', 'peso_kgf_m', 'peso_total_kgf',
               'UC_max', 'util_max', 'lam_max', 'lim_lam', 'local_ok', 'global_ok', 'cumple', 'barra_critica',
               'combo_critica', 'detalle_local', 'sap_actual']
def run_group(g):
    """g: fila de GRUPOS. Evalúa todos los candidatos de sus familias y ordena por peso total."""
    fam_ok = [s.strip() for s in g['familias'].split(',')]
    frames_g = fr[fr['grupo'] == g['grupo']]
    dem_g = dem[dem['grupo'] == g['grupo']]
    c_pool = cand[cand['tipo'].isin(fam_ok)]
    lim_g = LAM_GLOBAL_ARR if g['tipo_miembro'] == 'Arriostramiento' else float(g['lim_KLr'])
    rows = []
    sap_list = ', '.join(sorted(frames_g['AnalSect'].dropna().unique()))
    for _, c in c_pool.iterrows():
        c = c.to_dict()
        uc, det = check_section(c, frames_g, dem_g)
        if not np.isfinite(uc):
            continue
        Ca_max = 0.0
        if c['kind'] == 'I' and g['tipo_miembro'] != 'Arriostramiento':
            Ca_max = max(0.0, OMEGA_C * (-dem_g['P'].min()) / (FYE * c['A']))
        ok_loc, det_loc = check_local(c, g['tipo_miembro'], Ca_max)
        lam_max = det.get('lam_max', 0.0)
        ok_glob = lam_max <= lim_g
        if str(g.get('excepcion_0_7R1', 'No')).strip().lower() in ('sí', 'si', 'yes', 'true'):
            ok_loc, ok_glob = True, True          # 8.8.4: exceptuado (0,7·R1 ≥ 1,0); el usuario debe confirmarlo
            det_loc = {'nota': 'excepción 8.8.4 (0,7R1≥1,0) aplicada'}
        peso_tot = float((frames_g['L_mm'] / 1000 * c['peso']).sum()) if np.isfinite(c['peso']) else np.nan
        cumple = (uc <= g['util_max']) and ok_loc and ok_glob
        rows.append(dict(grupo=g['grupo'], tipo_miembro=g['tipo_miembro'], designacion=c['designacion'],
                         familia=c['familia'], tipo=c['tipo'], peso_kgf_m=round(c['peso'], 2),
                         peso_total_kgf=round(peso_tot, 1), UC_max=round(uc, 3), util_max=g['util_max'],
                         lam_max=lam_max, lim_lam=round(lim_g, 1), local_ok=ok_loc, global_ok=ok_glob,
                         cumple=cumple, barra_critica=det.get('frame'), combo_critica=det.get('combo'),
                         detalle_local=str(det_loc), sap_actual=sap_list))
    res = pd.DataFrame(rows, columns=RESULT_COLS)
    return res.sort_values(['cumple', 'peso_total_kgf'], ascending=[False, True])

RESULTADOS = {}
for _, g in GRUPOS.iterrows():
    RESULTADOS[g['grupo']] = run_group(g)
    r = RESULTADOS[g['grupo']]
    ok = r[r['cumple']]
    print(f"\n=== {g['grupo']} ({g['tipo_miembro']}) | candidatos evaluados: {len(r)} | cumplen: {len(ok)} | util_max={g['util_max']}")
    print('    SAP actual:', r['sap_actual'].iloc[0] if len(r) else '-')
    if len(ok):
        print(ok.head(8)[['designacion', 'peso_kgf_m', 'peso_total_kgf', 'UC_max', 'lam_max', 'lim_lam', 'local_ok', 'global_ok', 'barra_critica', 'combo_critica']].to_string(index=False))
    else:
        print('    Ningún perfil del catálogo cumple. Motivo más frecuente (UC > util_max o esbeltez):')
        print(r[['designacion', 'UC_max', 'local_ok', 'global_ok']].head(5).to_string(index=False))

os.makedirs(WORK_DIR, exist_ok=True)
pd.concat(RESULTADOS.values()).to_csv(os.path.join(WORK_DIR, 'resultados_optimos_por_grupo.csv'), index=False)
print('\nResultados guardados en', os.path.join(WORK_DIR, 'resultados_optimos_por_grupo.csv'))

In [ ]:
# %% [interactivo] LISTA DESPLEGABLE DE PERFILES QUE CUMPLEN, POR GRUPO
from IPython.display import display, HTML
try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

def resumen_perfil(row):
    txt = [f"<b>{row['designacion']}</b> ({row['tipo']}) — peso {row['peso_kgf_m']} kgf/m, total grupo {row['peso_total_kgf']} kgf",
           f"UC máx = {row['UC_max']} (límite {row['util_max']}) · barra crítica {row['barra_critica']} · combo {row['combo_critica']}",
           f"Esbeltez global máx = {row['lam_max']} (límite {row['lim_lam']}) · local: {row['detalle_local']}"]
    return '<br>'.join(txt)

def tabla_cumplen(res):
    ok = res[res['cumple']]
    return ok[['designacion', 'tipo', 'peso_kgf_m', 'peso_total_kgf', 'UC_max', 'lam_max', 'lim_lam']]

PANELES = {}
for grupo, res in RESULTADOS.items():
    ok = res[res['cumple']].reset_index(drop=True)
    if ok.empty:
        display(HTML(f'<h4>{grupo}</h4><p>Ningún perfil del catálogo cumple con los parámetros actuales.</p>'))
        continue
    if HAS_WIDGETS:
        opciones = [(f"{r.designacion}  |  {r.peso_total_kgf} kgf  |  UC {r.UC_max}", i) for i, r in ok.iterrows()]
        dd = widgets.Dropdown(options=opciones, description='Perfil:', layout=widgets.Layout(width='700px'))
        salida = widgets.Output()
        def _mostrar(change, ok=ok, salida=salida):
            with salida:
                salida.clear_output()
                display(HTML(resumen_perfil(ok.loc[change['new']])))
        dd.observe(_mostrar, names='value')
        with salida:
            display(HTML(resumen_perfil(ok.loc[0])))
        PANELES[grupo] = widgets.VBox([widgets.HTML(f'<h4>{grupo}: {len(ok)} perfiles cumplen (mejor primero)</h4>'), dd, salida])
        display(PANELES[grupo])
    else:
        display(HTML(f'<h4>{grupo}: {len(ok)} perfiles cumplen</h4>'))
        display(tabla_cumplen(res))


In [ ]:
# %% [desplazamiento] VERIFICACIÓN DE DESPLAZAMIENTO DE ENTREPISO (combinaciones sísmicas)
# ---- DATOS EDITABLES ----
NIVELES = pd.DataFrame([
    # nivel,     z_m (cota del nivel), tol_m (tolerancia para tomar nodos), H_m (altura de piso; vacío = diferencia de cotas), nudos (vacío = todos los nodos de la cota)
    ['Base',      0.00, 0.10, None, ''],
    ['Nivel 1',   2.84, 0.15, None, ''],
    ['Nivel 2',   5.62, 0.15, None, ''],
    ['Techo',     7.95, 0.25, None, ''],
], columns=['nivel', 'z_m', 'tol_m', 'H_m', 'nudos'])
DIVISOR_DRIFT = 66          # límite admisible: H/66 (NCh2369)
FACTOR_DESP = 1.0           # multiplicador de los desplazamientos del modelo (p. ej. Cd/I si el análisis es elástico)
AGREG_NIVEL = 'mean'        # 'mean' (diafragma rígido) o 'max' (máximo |U| entre nodos del nivel)
COMBOS_SISMICAS = None      # None = automático: combinaciones con Ex, Ey o Ez en el nombre. O lista de nombres.
DIRECCIONES = ['X', 'Y']

# ---- CÁLCULO ----
nod = pd.DataFrame(raw['JOINT COORDINATES'], columns=['Joint', 'X', 'Y', 'Z'])
dsp = pd.DataFrame(raw['JOINT DISPLACEMENTS'], columns=['Joint', 'combo', 'U1', 'U2', 'U3'])
if COMBOS_SISMICAS is None:
    seis = sorted({c for c in dsp['combo'].unique() if re.search(r'E[xyz]', c)})
else:
    seis = [c for c in COMBOS_SISMICAS if c in set(dsp['combo'])]
dsp = dsp[dsp['combo'].isin(seis)]
print('Combinaciones sísmicas:', len(seis), '|', seis[0] if seis else '-', '...', seis[-1] if seis else '-')
lvl_nodes = {}
for _, n in NIVELES.iterrows():
    if str(n['nudos']).strip():
        lista = [j.strip() for j in str(n['nudos']).split(',') if j.strip()]
        sel = nod[nod['Joint'].isin(lista)]
    else:
        sel = nod[(nod['Z'] - n['z_m']).abs() <= n['tol_m']]
    assert len(sel) > 0, f"El nivel '{n['nivel']}' no tiene nodos (revisa z_m y tol_m)"
    lvl_nodes[n['nivel']] = sel['Joint'].tolist()
    print(f"{n['nivel']:>10}: z = {n['z_m']} m, {len(sel)} nodos")
H = []
zs = NIVELES['z_m'].tolist()
for k, n in NIVELES.iterrows():
    if pd.notna(n['H_m']):
        H.append(float(n['H_m']))
    elif k == 0:
        H.append(np.nan)
    else:
        H.append(zs[k] - zs[k - 1])
NIVELES['H_usada_m'] = H
filas = []
for combo in seis:
    dc = dsp[dsp['combo'] == combo].set_index('Joint')
    for d, col in (('X', 'U1'), ('Y', 'U2')):
        if d not in DIRECCIONES:
            continue
        val = []
        for _, n in NIVELES.iterrows():
            u = dc.loc[[j for j in lvl_nodes[n['nivel']] if j in dc.index], col]
            val.append(float(u.mean()) if AGREG_NIVEL == 'mean' else float(u.abs().max() * np.sign(u.loc[u.abs().idxmax()])))
        for k in range(1, len(NIVELES)):
            n = NIVELES.iloc[k]
            delta = abs(val[k] - val[k - 1]) * FACTOR_DESP
            h = NIVELES.iloc[k]['H_usada_m']
            ratio = delta / h
            filas.append(dict(combo=combo, direccion=d, piso=n['nivel'], H_m=round(h, 3), delta_mm=round(delta * 1000, 2),
                              deriva=round(ratio, 6), limite=round(1 / DIVISOR_DRIFT, 6), cumple=ratio <= 1 / DIVISOR_DRIFT))
DRIFT = pd.DataFrame(filas)
RESUMEN_DRIFT = (DRIFT.sort_values('deriva', ascending=False)
                 .groupby(['combo', 'direccion'], as_index=False).first()
                 .sort_values('deriva', ascending=False))
RESUMEN_DRIFT['cumple_todas'] = RESUMEN_DRIFT['cumple']
print(f"\nLímite de deriva: H/{DIVISOR_DRIFT} = {1/DIVISOR_DRIFT:.5f} · factor de desplazamiento = {FACTOR_DESP}")
print('Peor caso por combinación y dirección (ordenado por deriva):')
print(RESUMEN_DRIFT[['combo', 'direccion', 'piso', 'H_m', 'delta_mm', 'deriva', 'cumple']].head(15).to_string(index=False))
print(f"\nCombinaciones que NO cumplen: {int((~RESUMEN_DRIFT['cumple']).sum())} de {len(RESUMEN_DRIFT)}")
DRIFT.to_csv(os.path.join(WORK_DIR, 'derivas_entrepiso.csv'), index=False)
